# Env setup

In [2]:
import os
os.environ["OPENROUTER_API_KEY"] = "..."
os.environ["HF_API_KEY"] = "..."

In [ ]:
"""
VLA-0 LIBERO Evaluation Setup
==============================
End-to-end setup script: system deps → cloning → installs →
dataloader patch → model download → compatibility patches → smoke-test.

Run on Kaggle GPU (T4/P100) with ~30 GB disk and EGL/CUDA available.
"""

import subprocess, sys, os, re, pathlib

# ─────────────────────────────────────────────
# 0. Helpers
# ─────────────────────────────────────────────

def bash(cmd: str) -> None:
    """Run a shell command, streaming output, raise on failure."""
    result = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}):\n{cmd}")


# ─────────────────────────────────────────────
# 1. System dependencies
# ─────────────────────────────────────────────

bash("""
apt-get update -qq
apt-get install -y -qq \
    libgl1-mesa-glx libgl1-mesa-dri \
    libegl1-mesa libegl-dev \
    libgles2-mesa libgles2-mesa-dev \
    libosmesa6 libosmesa6-dev \
    libglfw3 libglfw3-dev \
    patchelf ffmpeg git git-lfs \
    build-essential python3-dev
echo "System deps installed"
""")


# ─────────────────────────────────────────────
# 2. Environment variables (EGL headless rendering)
# ─────────────────────────────────────────────

os.environ["MUJOCO_GL"]         = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["DISPLAY"]           = ""
print("Env vars set")


# ─────────────────────────────────────────────
# 3. Clone VLA-0 (with submodules)
# ─────────────────────────────────────────────

REPO_DIR = "/kaggle/working/vla0"

bash(f"""
if [ ! -d "{REPO_DIR}/.git" ]; then
    git clone --recurse-submodules https://github.com/NVlabs/vla0.git "{REPO_DIR}"
    echo "Cloned."
else
    echo "Repo exists — ensuring submodules are initialised..."
    git -C "{REPO_DIR}" submodule update --init --recursive
    echo "Submodules OK."
fi
""")


# ─────────────────────────────────────────────
# 4. Python package installs
# ─────────────────────────────────────────────

bash(f"""
set -e

pip install -q --upgrade pip setuptools wheel hatchling hatch-vcs build

echo "--- vla0 ---"
pip install -q -e {REPO_DIR}

echo "--- bundled lerobot (no-deps) ---"
pip install -q --no-deps --no-build-isolation \
    -e {REPO_DIR}/libs/RoboVerse/libs/lerobot

echo "--- RoboVerse ---"
pip install -q --no-build-isolation -e {REPO_DIR}/libs/RoboVerse

echo "--- egl_probe (pre-build) ---"
pip install -q --no-build-isolation egl_probe hf-egl-probe

echo "--- LIBERO ---"
pip install -q --no-build-isolation libero

echo "--- extras ---"
pip install -q tbparse tensorboardX qwen-vl-utils \
    "imageio-ffmpeg>=0.5.1" "imageio>=2.35.0" "setuptools>=70.0"

echo "--- restore Kaggle torch stack ---"
pip install -q \
    "torch==2.10.0+cu128" \
    "torchvision==0.25.0+cu128" \
    "torchaudio==2.10.0+cu128" \
    --extra-index-url https://download.pytorch.org/whl/cu128

echo "========== All installs done =========="
""")


# ─────────────────────────────────────────────
# 5. Sys-path setup
# ─────────────────────────────────────────────

for p in [
    REPO_DIR,
    f"{REPO_DIR}/libs/LIBERO",
    f"{REPO_DIR}/libs/RoboVerse",
    f"{REPO_DIR}/libs/RoboVerse/libs/lerobot/src",   # bundled lerobot first
]:
    if p not in sys.path:
        sys.path.insert(0, p)


# ─────────────────────────────────────────────
# 6. Dataloader patch (MultiLeRobotDataset compatibility)
# ─────────────────────────────────────────────

DATALOADER = pathlib.Path(
    f"{REPO_DIR}/libs/RoboVerse/roboverse/datasets/lerobot/dataloader.py"
)
MARKER = "# @@PATCHED@@"
src = DATALOADER.read_text()

if MARKER in src:
    print("Dataloader already patched ✓")
else:
    pattern = re.compile(
        r"try\s*:\s*\n"
        r"(?:.*\n)*?.*MultiLeRobotDataset\)[ \t]*\n"
        r"except ImportError\s*:\s*\n"
        r"(?:.*\n)*?.*MultiLeRobotDataset\)[ \t]*",
        re.MULTILINE,
    )
    REPLACEMENT = """\
# @@PATCHED@@
# Works with any lerobot version — stubs MultiLeRobotDataset if absent
def _import_lerobot():
    import importlib
    for _p in ["lerobot.datasets.lerobot_dataset",
               "lerobot.common.datasets.lerobot_dataset"]:
        try:
            _m = importlib.import_module(_p)
            _DS    = getattr(_m, "LeRobotDataset")
            _Meta  = getattr(_m, "LeRobotDatasetMetadata")
            _Multi = getattr(_m, "MultiLeRobotDataset", None)
            if _Multi is None:
                class _Multi:
                    def __init__(self, *a, **kw):
                        raise NotImplementedError(
                            "MultiLeRobotDataset removed in this lerobot version.")
            return _DS, _Meta, _Multi
        except (ImportError, ModuleNotFoundError):
            continue
    raise ImportError("lerobot dataset module not found on any known path")

LeRobotDataset, LeRobotDatasetMetadata, MultiLeRobotDataset = _import_lerobot()"""

    patched, n = pattern.subn(REPLACEMENT, src)
    if n == 0:
        print("⚠️  Dataloader regex didn't match — printing first 80 lines:")
        for i, line in enumerate(src.splitlines()[:80], 1):
            print(f"{i:3}: {line}")
    else:
        DATALOADER.write_text(patched)
        for pyc in DATALOADER.parent.glob("__pycache__/dataloader*.pyc"):
            pyc.unlink()
        print(f"Dataloader patched ({n} occurrence(s)) ✓")


# # ─────────────────────────────────────────────
# # 7. Download model checkpoint
# # ─────────────────────────────────────────────

# bash(f"""
# pip install huggingface_hub -q
# python - <<'EOF'
# from huggingface_hub import snapshot_download
# import os

# out_dir = "{REPO_DIR}/runs/vla0"
# os.makedirs(out_dir, exist_ok=True)

# path = snapshot_download(
#     repo_id="ankgoyal/vla0-libero",
#     local_dir=out_dir,
#     allow_patterns=[
#         "model_last/*",       # HF-format checkpoint folder
#         "config.yaml",        # training config
#         "dataset_stats.pkl",  # loaded by set_dataset_stats
#     ],
# )
# print(f"Downloaded to: {{path}}")
# EOF
# """)


# ─────────────────────────────────────────────
# 8. LeRobot compatibility patches
# ─────────────────────────────────────────────

import lerobot.datasets.lerobot_dataset as le_ds
import lerobot.datasets.utils as le_utils

# 8a. get_safe_version → always return "main" (skips hub version negotiation)
def _safe_version_noop(repo_id, version):
    return "main"

le_ds.get_safe_version = _safe_version_noop
le_utils.get_safe_version = _safe_version_noop

# 8b. check_version_compatibility → no-op
def _check_version_noop(repo_id, version_to_check, current_version,
                         enforce_breaking_major=True):
    pass

le_ds.check_version_compatibility = _check_version_noop
le_utils.check_version_compatibility = _check_version_noop

# 8c. BackwardCompatibilityError → non-raising subclass
import lerobot.datasets.backward_compatibility as le_bc

class _SafeBackwardCompatibilityError(Exception):
    def __init__(self, *args, **kwargs):
        super().__init__("Backward compatibility issue (suppressed)")

le_bc.BackwardCompatibilityError  = _SafeBackwardCompatibilityError
le_ds.BackwardCompatibilityError  = _SafeBackwardCompatibilityError
le_utils.BackwardCompatibilityError = _SafeBackwardCompatibilityError

print("LeRobot patches applied ✓")


# ─────────────────────────────────────────────
# 9. RoboVerse metadata patch
# ─────────────────────────────────────────────

import roboverse.datasets.lerobot.dataloader as rv_dl

class MockLeRobotMetadata:
    @property
    def camera_keys(self):
        return ["image", "wrist_image"]

rv_dl.get_lerobot_metadata = lambda repo_id: MockLeRobotMetadata()
print("RoboVerse get_lerobot_metadata patched ✓")


# ─────────────────────────────────────────────
# 10. Smoke test
# ─────────────────────────────────────────────

import unittest.mock

os.chdir(REPO_DIR)

with unittest.mock.patch("builtins.input", return_value="N"):
    import lerobot
    print(f"lerobot  : {lerobot.__file__}")

    import torch
    print(f"torch    : {torch.__version__}")

    from libero.libero import benchmark
    from libero.libero.envs import OffScreenRenderEnv
    print("LIBERO OK")

    from roboverse.evals.libero.eval import get_evaluation_tasks
    tasks = get_evaluation_tasks()
    print(f"RoboVerse OK — {len(tasks)} task entries found")

print("\n✅  All setup steps complete. Ready to run VLA-0 evaluation.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../00-libpython3.10-dev_3.10.12-1~22.04.15_amd64.deb ...
Unpacking libpython3.10-dev:amd64 (3.10.12-1~22.04.15) over (3.10.12-1~22.04.14) ...
Preparing to unpack .../01-libpython3.10_3.10.12-1~22.04.15_amd64.deb ...
Unpacking libpython3.10:amd64 (3.10.12-1~22.04.15) over (3.10.12-1~22.04.14) ...
Preparing to unpack .../02-python3.10_3.10.12-1~22.04.15_amd64.deb ...
Unpacking python3.10 (3.10.12-1~22.04.15) over (3.10.12-1~22.04.14) ...
Preparing to unpack .../03-libpython3.10-stdlib_3.10.12-1~22.04.15_amd64.deb ...
Unpacking libpython3.10-stdlib:amd64 (3.10.12-1~22.04.15) over (3.10.12-1~22.04.14) ...
Preparing to unpack .../04-python3.10-minimal_3.10.12-1~22.04.15_amd64.deb ...
Unpacking python3.10-minimal (3.10.12-1~22.04.15) over (3.10.12-1~22.04.14) ...
Preparing to unpack .../05-libpython3.10-minimal_3.10.12-1~22.04.15_amd64.deb ...
Unpacking libpython3.10-minimal:amd64 (3.10.12-1~22.04.1

### checkpoints download

In [ ]:
%%bash
pip install huggingface_hub -q
python - <<'EOF'
from huggingface_hub import snapshot_download
import os

out_dir = "/kaggle/working/vla0/runs/vla0"
os.makedirs(out_dir, exist_ok=True)

path = snapshot_download(
    repo_id="ankgoyal/vla0-libero",
    local_dir=out_dir,
    allow_patterns=["model_last.pth"],
)
print(f"Downloaded to: {path}")
EOF


In [ ]:
import torch
import os

pth_path = "/kaggle/working/vla0/runs/vla0/model_last.pth"

print("Loading .pth into RAM...")
checkpoint_cache = torch.load(pth_path, map_location="cpu", weights_only=False)
print("Loaded into RAM!")

os.remove(pth_path)
print(f"Deleted {pth_path} from disk, freed ~15GB")

_original_torch_load = torch.load
def _patched_torch_load(f, *args, **kwargs):
    if isinstance(f, str) and f == pth_path:
        print(f"Intercepted torch.load({f}) → returning from RAM")
        return checkpoint_cache
    return _original_torch_load(f, *args, **kwargs)

torch.load = _patched_torch_load
print("Monkey-patched torch.load ✓")

In [ ]:
%%bash
python - <<'EOF'
from huggingface_hub import snapshot_download
import os

out_dir = "/kaggle/working/vla0/runs/vla0"
os.makedirs(out_dir, exist_ok=True)

path = snapshot_download(
    repo_id="ankgoyal/vla0-libero",
    local_dir=out_dir,
    allow_patterns=[
        "model_last/*",
        "config.yaml",
        "dataset_stats.pkl",
    ],
)
print(f"Downloaded to: {path}")
EOF

In [ ]:
import lerobot
print(f"lerobot  : {lerobot.__file__}")

import torch
print(f"torch    : {torch.__version__}")

from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv
print("LIBERO OK")

from roboverse.evals.libero.eval import get_evaluation_tasks
tasks = get_evaluation_tasks()
print(f"RoboVerse OK — {len(tasks)} task entries found")

# Models

In [ ]:
# Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
#
# Licensed under the CC BY-NC 4.0 license [see LICENSE for details].

import gc
import random
import warnings
from typing import List

import numpy as np
import PIL
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info
from torch import nn
from transformers import AutoConfig, LogitsProcessor, Qwen2_5_VLProcessor
from transformers.models.qwen2_5_vl.modeling_qwen2_5_vl import \
    Qwen2_5_VLForConditionalGeneration

import rv_train.constants as C
# from rv_train.utils.train_utils import ForkedPdb as debug  # noqa: F401


class NumberSpaceOnlyProcessor(LogitsProcessor):
    """
    Logits processor that constrains generation to only numbers (0-9), spaces, and end-of-text tokens.
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        # Get token IDs for allowed tokens
        self.allowed_tokens = set()

        # Add numbers 0-9
        for i in range(10):
            token_id = tokenizer.encode(str(i), add_special_tokens=False)[0]
            self.allowed_tokens.add(token_id)
        # Add space token
        space_token_id = tokenizer.encode(" ", add_special_tokens=False)[0]
        self.allowed_tokens.add(space_token_id)

        # Add end of text token
        if tokenizer.eos_token_id is not None:
            self.allowed_tokens.add(tokenizer.eos_token_id)

    def __call__(self, input_ids, scores):
        # Set logits to negative infinity for all tokens except allowed ones
        mask = torch.full_like(scores, float("-inf"))
        for token_id in self.allowed_tokens:
            mask[:, token_id] = 0
        return scores + mask


def format_data(system_message, image, instr, action_txt):
    """
    Convert the data into the format required by the model
    :param system_message: str, the system message
    :param image: list of PIL images, the image to be processed
    :param instr: str, the instruction
    :param action_txt: str, the action text
    :return: list of dicts, the format required by the model
    """
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": system_message}],
        },
        {
            "role": "user",
            "content": [{"type": "image", "image": _image} for _image in image]
            + [{"type": "text", "text": instr}],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": action_txt}],
        },
    ]




class VLA0_QwenActor(nn.Module):
    def __init__(
        self,
        qwen_model_id,
        action_type,
        original_action_dim,
        horizon,
        history,
        use_lora=True,
        use_qlora=True,
        num_cam=1,
        lora_config="",
        lora_rank=8,
        rgb_input=False,
        rgb_img_size=(84, 84),
        add_vision_id=False,
        tiled_rgb_imgs=False,
        num_bins_actions=1000,
        use_flash_attention_2=False,
        system_message_version=1,
        action_mask_aug=0,
        action_mask_aug_per=0.1,
        attention_dropout=0.0,
    ):
        """
        :param qwen_model_id: str, the id of the qwen model to use
        :param action_type: str, the type of action to use, either ORIGINAL or EE
        :param original_action_dim: int, the dimension of the original action
        :param horizon: int, the horizon of the action
        :param history: int, the history of the action
        :param use_qlora: bool, whether to use qlora for parameter efficient fine-tuning
        :param num_cam: int, the number of cameras for rgb input
        :param lora_config: str, the lora configuration to use, either empty string or "default"
        :param lora_rank: int, the rank of the lora to use, only used if lora_config is "default"
        :param rgb_input: bool, whether to use rgb image input
        :param rgb_img_size: tuple, the size of the rgb image input (height, width)
        :param add_vision_id: bool, whether to add vision id to the input for qwen2.5
        :param tiled_rgb_imgs: bool, whether to tile the rgb images into a single image instead feeding them separately
        :param num_bins_actions: int, the number of bins in which each action dimension is discretized
        :param use_flash_attention_2: bool, whether to use flash attention 2 for faster training and inference
        :param attention_dropout: float, the dropout rate for the attention layer in the qwen model. Only tested when use_lora is False.
        """
        super().__init__()

        # current assumptions
        if use_qlora:
            assert use_lora, "use_lora must be True if use_qlora is True"
        if attention_dropout > 0.0:
            assert (
                not use_lora
            ), "attention_dropout is only supported when use_lora is False"
        assert lora_config in ["", "default"]
        if history > 1 or num_cam > 1:
            assert (
                add_vision_id
            ), "add_vision_id must be True if history > 1 or num_cam > 1"

        # assert not use_flash_attention_2, "use_flash_attention_2 is not supported yet, it requires tokenizer.pad=left which we have not fully implemented/understood"

        # for Qwen model, we need to load the parameters before DDP
        # in case we want to load the model from a checkpoint
        self.load_param_before_ddp = True

        self.qwen_model_id = qwen_model_id
        self.action_type = action_type
        self.original_action_dim = original_action_dim
        self.horizon = horizon
        self.history = history
        self.use_lora = use_lora
        self.use_qlora = use_qlora
        self.num_cam = num_cam
        self.lora_config = lora_config
        self.lora_rank = lora_rank
        self.rgb_input = rgb_input
        self.rgb_img_size = rgb_img_size
        self.add_vision_id = add_vision_id
        self.tiled_rgb_imgs = tiled_rgb_imgs
        self.num_bins_actions = num_bins_actions
        self.use_flash_attention_2 = use_flash_attention_2
        self.action_mask_aug_per = action_mask_aug_per
        self.attention_dropout = attention_dropout

        self.model = self.load_qwen_model(
            qwen_model_id=self.qwen_model_id,
            use_lora=self.use_lora,
            use_qlora=self.use_qlora,
            lora_config=self.lora_config,
            lora_rank=self.lora_rank,
            use_flash_attention_2=self.use_flash_attention_2,
            attention_dropout=self.attention_dropout,
        )

        # Enable gradient checkpointing if requested
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

        self.min_pixel = self.max_pixel = self.rgb_img_size[0] * self.rgb_img_size[1]
        if self.rgb_input and self.tiled_rgb_imgs:
            self.min_pixel *= self.history * self.num_cam
            self.max_pixel *= self.history * self.num_cam

        self.processor = self.load_qwen_model_processor(
            qwen_model_id=qwen_model_id,
            min_pixel=self.min_pixel,
            max_pixel=self.max_pixel,
            padding_side="left" if use_flash_attention_2 else None,
        )
        self.logits_processor = NumberSpaceOnlyProcessor(self.processor.tokenizer)

        print(
            "WARNING: Using hardcoded dataset stats for DP3. This should be replaced with loading from a file."
        )

        if action_type == C.ORIGINAL:
            self.act_dim = original_action_dim
        elif action_type == C.EE:
            self.act_dim = 7
        else:
            assert False

        self.system_message = f"Analyze the input image and predict robot actions for the next {self.horizon} timesteps. Each action has {self.act_dim} dimensions. Output a single sequence of {self.horizon * self.act_dim} integers (0-{self.num_bins_actions} each), representing the {self.horizon} timesteps sequentially. Provide only space separated numbers. Nothing else."

        # todo: need better way to determine it
        self.num_tokens = 1024
        self.original_dataset_stats = None  # original dataset stats has the same format as the one provided by the dataset
        self.dataset_stats = (
            None  # dataset stats is in the format specific to the model
        )

        self._sysuser_len = None

        self.cache_sysuser_len = False

    def set_dataset_stats(self, dataset_stats):
        """
        Set the dataset stats for the model
        :param dataset_stats: dict, the dataset stats
        """
        if dataset_stats == {}:
            warnings.warn(
                "Dataset stats is empty likely because the system does not have the data used to compute the stats. Ignore this is you are loading a pretrained model."
            )
            return

        self.original_dataset_stats = dataset_stats
        if self.action_type == C.ORIGINAL:
            self.dataset_stats = dataset_stats["out_ori_act"]
        else:
            raise NotImplementedError(f"Action type {self.action_type} not implemented")

    @staticmethod
    def load_qwen_model(
        qwen_model_id,
        use_lora,
        use_qlora,
        lora_config,
        lora_rank,
        use_flash_attention_2,
        attention_dropout,
    ):
        if lora_config == "":
            lora_config = None
        elif lora_config == "default":
            if use_lora:
                from peft import LoraConfig, get_peft_model

                lora_config = LoraConfig(
                    lora_alpha=16,
                    lora_dropout=0.05,
                    r=lora_rank,
                    bias="none",
                    target_modules=["q_proj", "v_proj"],
                    task_type="CAUSAL_LM",
                )
        else:
            raise ValueError(f"Invalid lora_config: {lora_config}")

        # Qwen Init
        bnb_config = None
        if use_lora and use_qlora:
            from transformers import BitsAndBytesConfig

            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_type=torch.bfloat16,
            )

        extra_kwargs = {}
        if use_flash_attention_2:
            extra_kwargs["attn_implementation"] = "flash_attention_2"

        if attention_dropout > 0.0:
            config = AutoConfig.from_pretrained(qwen_model_id)
            config.attention_dropout = attention_dropout
            extra_kwargs["config"] = config

        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            qwen_model_id,
            # device_map={"": "cuda:0"},  # Use the explicit map
            quantization_config=bnb_config,
            torch_dtype=torch.bfloat16,
            # local_files_only=True,
            **extra_kwargs,
        )

        if use_lora and (lora_config is not None):
            model = get_peft_model(model, lora_config)

        return model

    @staticmethod
    def load_qwen_model_processor(
        qwen_model_id,
        min_pixel,
        max_pixel,
        padding_side,
    ):
        if padding_side is not None:
            processor = Qwen2_5_VLProcessor.from_pretrained(
                qwen_model_id,
                min_pixels=min_pixel,
                max_pixels=max_pixel,
                padding_side=padding_side,
                # local_files_only=True,
            )
        else:
            processor = Qwen2_5_VLProcessor.from_pretrained(
                qwen_model_id,
                min_pixels=min_pixel,
                max_pixels=max_pixel,
                # local_files_only=True,
            )

        return processor

    def get_min_max_act(self, instruction):
        """
        Get the min and max action for the instruction.
        :param instruction: str, the instruction for the current episode. This is needed for libero bounds 99% as the action space is different for different instructions.
        :return: torch.Tensor, the min and max action.
        """
        assert instruction is not None, "instruction is needed for libero bounds 99%"
        min_act = []
        max_act = []
        for _instruction in instruction:
            _suite = self.instruction_to_suite[_instruction]
            min_act.append(torch.tensor(self.dataset_stats[_suite]["min"]))
            max_act.append(torch.tensor(self.dataset_stats[_suite]["max"]))
        min_act = torch.stack(min_act, dim=0)
        max_act = torch.stack(max_act, dim=0)
        return min_act, max_act

    def get_text_action(self, actions, instruction=None):
        """
        Get the text action from the actions.
        :param actions: torch.Tensor, the actions to convert to text.
        :param instruction: str, the instruction for the current episode. This is needed for libero bounds 99% as the action space is different for different instructions.
        :return: List[str], the text action.
        """
        # TODO: implement this for ee action space
        if (
            (not hasattr(self, "_min_act"))
            or (not hasattr(self, "_max_act"))
            or (self._min_act.device != actions.device)
            or (self._max_act.device != actions.device)
        ):
            # safer to recompute the min and max action for each device
            min_act = torch.tensor(self.dataset_stats["min"], device=actions.device)
            max_act = torch.tensor(self.dataset_stats["max"], device=actions.device)
            self._min_act = min_act
            self._max_act = max_act
        else:
            # use the cached min and max action
            min_act = self._min_act
            max_act = self._max_act

        assert torch.all(min_act <= actions) and torch.all(
            actions <= max_act
        ), f"Action is out of range: {actions}"

        actions = (actions - min_act) / (max_act - min_act)
        actions *= self.num_bins_actions
        actions = torch.round(actions).long()
        actions = actions.reshape(actions.shape[0], -1)
        action_txt = [" ".join(map(str, x.tolist())) for x in actions]

        return action_txt

    def get_action_from_text_action(self, action_txt, instruction=None):
        """
        Get the action from the text action.
        :param action_txt: List[str], the text action.
        :param instruction: str, the instruction for the current episode. This is needed for libero bounds 99% as the action space is different for different instructions.
        """
        # TODO: implement this for ee action space
        bs = len(action_txt)
        min_act = torch.tensor(self.dataset_stats["min"])
        max_act = torch.tensor(self.dataset_stats["max"])

        try:
            """ "
            Note: The action_txt is a list of strings. action_txt[i] is the action text for the i-th sample.
            action_txt[i] is a string that contains horizon * act_dim numbers in space separated format.
            We have built in some flexbility for handling minor mistakes in the action_txt.
            """
            # remove space from the front and back of the action_txt if they exist
            action_txt = [x.strip() for x in action_txt]
            action = [[x for x in _action_txt.split(" ")] for _action_txt in action_txt]
            action = torch.tensor(
                [[int(x) for x in _action_txt] for _action_txt in action],
                dtype=torch.float32,
            )
            # This handles tha case when bs == 1 and the action_txt is not divisible by act_dim
            # We remove some elements so that it is divisible by act_dim
            if bs == 1 and len(action[0]) % self.act_dim != 0:
                action = action[0][: len(action[0]) - len(action[0]) % self.act_dim][
                    None, :
                ]

            # reshape to (bs, -1, act_dim)
            # takes care of case when the action_txt has less than horizon * act_dim numbers
            action = action.reshape(bs, -1, self.act_dim)
            # if action.shape[1] < self.horizon, pad the action with the last action
            if action.shape[1] < self.horizon:
                action = torch.cat(
                    [
                        action,
                        action[:, -1:].repeat(1, self.horizon - action.shape[1], 1),
                    ],
                    dim=1,
                )
            if action.shape[1] > self.horizon:
                action = action[:, : self.horizon]
            action = ((action / self.num_bins_actions) * (max_act - min_act)) + min_act
        except Exception as e:
            print(f"Error in parsing action text: {e}")
            print(action_txt)
            action = ((min_act + max_act) / 2).repeat(bs, self.horizon, 1)

        return action

    def check_inputs(
        self,
        pc,
        rgb_pc,
        instr,
        rgb,
        ori_act,
        ee_pos,
        ee_rot,
        ee_gri,
        out_ee_pos,
        out_ee_rot,
        out_ee_gri,
        out_ori_act,
        get_loss,
        get_action,
        get_one_step_action,
        last_action_txt,
    ):
        assert (
            self.dataset_stats is not None
        ), "dataset_stats must be set before calling forward"
        assert isinstance(instr, list)

        assert not (get_loss and get_action)
        assert get_loss or get_action
        if get_one_step_action:
            assert get_action
            assert isinstance(last_action_txt, str)
            assert len(instr) == 1, "one_step_action is only supported for batch size 1"
        if self.rgb_input:
            assert rgb is not None
            assert rgb.shape[1:] == (
                self.history,
                self.num_cam,
                *self.rgb_img_size,
                3,
            ), f"rgb.shape: {rgb.shape}, self.history: {self.history}, self.num_cam: {self.num_cam}, self.rgb_img_size: {self.rgb_img_size}"
            # some room for numerical errors
            # 0, 2, 255 are valid values
            assert (rgb.min() >= -1e-2) and (
                1.99 <= rgb.max() <= 255.01
            ), f"rgb.min(): {rgb.min()}, rgb.max(): {rgb.max()}"
        else:
            assert rgb is None

        assert pc is None
        assert rgb_pc is None

    def get_imgs(
        self,
        bs,
        pc,
        rgb_pc,
        rgb,
    ):
        """
        Get the images for the given inputs
        :param bs: int, the batch size
        :param pc: torch.Tensor, the point cloud
        :param rgb_pc: torch.Tensor, the rgb point cloud
        :param rgb: torch.Tensor, the rgb image
        :return: list of list of PIL images
        """
        imgs = [[] for _ in range(bs)]

        if self.rgb_input:
            for i, _rgb in enumerate(rgb):
                _imgs = []
                for j in range(self.history):
                    for k in range(self.num_cam):
                        _imgs.append(_rgb[j][k])
                if self.tiled_rgb_imgs:
                    _imgs = [self.tile_images(_imgs)]
                _imgs = [
                    Image.fromarray(x.cpu().numpy().astype(np.uint8)) for x in _imgs
                ]
                imgs[i].extend(_imgs)

        return imgs

    def get_qwen_inputs(
        self,
        bs: int,
        imgs: List[List[PIL.Image.Image]],
        instr: List[str],
        action_txt: List[str],
        drop_assistant: bool = False,
        add_generation_prompt: bool = False,
    ):
        """
        Get the Qwen inputs for the given inputs
        :param bs: int, the batch size
        :param imgs: list of list of PIL images
        :param instr: list of strings
        :param action_txt: list of strings
        :param drop_assistant: bool, whether to drop the assistant portion.
        :param add_generation_prompt: bool, whether to add the generation prompt assistant\n.
        """

        examples = [
            format_data(
                system_message=self.system_message,
                image=imgs[i],
                instr=instr[i],
                action_txt=action_txt[i],
            )
            for i in range(bs)
        ]
        if drop_assistant:
            # drop the assistant portion so the model must generate it
            examples = [e[:2] for e in examples]

        texts = [
            self.processor.apply_chat_template(
                example,
                tokenize=False,
                add_generation_prompt=add_generation_prompt,
                add_vision_id=self.add_vision_id,
            )
            # when add_generation_prompt is True, it will add the prompt
            # `assistant\n` to the end of the input text
            for example in examples
        ]
        # [0] in process_vision_info is for image input, [1] is for video input
        image_inputs = [process_vision_info(example)[0] for example in examples]

        model_inputs = self.processor(
            text=texts, images=image_inputs, return_tensors="pt", padding=True
        )
        for key in model_inputs:
            model_inputs[key] = model_inputs[key].to(
                next(self.model.parameters()).device
            )
        return model_inputs, examples

    def forward(
        self,
        pc=None,
        rgb_pc=None,
        instr=None,
        rgb=None,
        ori_act=None,
        ee_pos=None,
        ee_rot=None,
        ee_gri=None,
        out_ee_pos=None,
        out_ee_rot=None,
        out_ee_gri=None,
        out_ori_act=None,
        get_loss=True,
        get_action=False,
        generate_temperature=0.1,
        get_one_step_action=False,
        last_action_txt="",
    ):
        """
        Forward pass for the Qwen model

        Parameters
        ----------
        pc : optional
            Point cloud data
        rgb_pc : optional
            RGB point cloud data
        instr : optional
            Instructions
        rgb : optional
            RGB images
        ori_act : optional
            Original actions
        ee_pos : optional
            End effector position
        ee_rot : optional
            End effector rotation
        ee_gri : optional
            End effector gripper state
        out_ee_pos : optional
            Output end effector position
        out_ee_rot : optional
            Output end effector rotation
        out_ee_gri : optional
            Output end effector gripper state
        out_ori_act : optional
            Output original actions
        get_loss : bool, default=True
            Whether to calculate and return loss
        get_action : bool, default=False
            Whether to get actions
        generate_temperature : float, default=0.1
            Temperature for generation
        get_one_step_action : bool, default=False
            Whether to run the model forward for only one step of action at a time. If True,
            we complete the last_action_txt to next action string sufficient to decode the
            action for one next step. If False, we complete the last_action_txt to next
            action string sufficient to decode the action for all the next steps.
        last_action_txt : str, default=""
            The last action text to complete to get the next action text. This is only
            used when get_one_step_action is True.

        Returns
        -------
        dict or tuple
            Model outputs (should specify exact return structure)

        Raises
        ------
        Exception
            Any exceptions that can be raised (if applicable)
        """
        self.check_inputs(
            pc=pc,
            rgb_pc=rgb_pc,
            instr=instr,
            rgb=rgb,
            ori_act=ori_act,
            ee_pos=ee_pos,
            ee_rot=ee_rot,
            ee_gri=ee_gri,
            out_ee_pos=out_ee_pos,
            out_ee_rot=out_ee_rot,
            out_ee_gri=out_ee_gri,
            out_ori_act=out_ori_act,
            get_loss=get_loss,
            get_action=get_action,
            get_one_step_action=get_one_step_action,
            last_action_txt=last_action_txt,
        )

        bs = len(instr)

        # imgs is list of list of PIL images
        imgs = self.get_imgs(
            bs=bs,
            pc=pc,
            rgb_pc=rgb_pc,
            rgb=rgb,
        )

        # TODO: implement this for ee action space
        if out_ori_act is None:
            assert not get_loss
            action_txt = [[]] * bs
        else:
            action_txt = self.get_text_action(out_ori_act, instruction=instr)

        model_inputs, examples = self.get_qwen_inputs(
            bs=bs,
            imgs=imgs,
            instr=instr,
            action_txt=action_txt,
            drop_assistant=get_action,  # when getting action, we drop the assistant portion
            add_generation_prompt=get_action,  # when getting action, we add the generation prompt assistant\n so that the model need not generate it
        )

        if get_loss:
            labels = model_inputs["input_ids"].clone()
            # mask system message and image token IDs in the labels
            for i, example in enumerate(examples):
                if (self._sysuser_len is None) or (not self.cache_sysuser_len):
                    sysuser_conv = example[:-1]
                    sysuser_text = self.processor.apply_chat_template(
                        sysuser_conv, tokenize=False, add_vision_id=self.add_vision_id
                    )
                    sysuser_img, _ = process_vision_info(sysuser_conv)

                    sysuser_inputs = self.processor(
                        text=[sysuser_text],
                        images=[sysuser_img],
                        return_tensors="pt",
                        padding=True,
                    )

                    sysuser_len = sysuser_inputs["input_ids"].shape[1]
                    sysuser_len += 3  # to mask out `assistant\n`
                    self._sysuser_len = sysuser_len
                else:
                    sysuser_len = self._sysuser_len
                # TIP: to decode the input use:
                # when padding is right: self.processor.decode(model_inputs["input_ids"][0][0:sysuser_len])
                # when padding is left: self.processor.decode(model_inputs["input_ids"][0][num_pad_tokens: num_pad_tokens + sysuser_len])
                if self.processor.tokenizer.padding_side == "right":
                    labels[i, :sysuser_len] = -100
                elif self.processor.tokenizer.padding_side == "left":
                    num_pad_tokens = sum(labels[i] == 151643).item()
                    labels[i, num_pad_tokens : num_pad_tokens + sysuser_len] = -100
                else:
                    raise ValueError(
                        f"Unknown padding side: {self.processor.tokenizer.padding_side}"
                    )

                assert (
                    not self.processor.tokenizer.padding_side == "left"
                ), "current implementation only supports right padding"
                # for debugging, compare
                # self.processor.decode(model_inputs["input_ids"][i][model_inputs["attention_mask"][i] == 1])
                # with self.processor.decode(model_inputs["input_ids"][i])
                _action_txt = action_txt[i]
                # 10% of sample has no augmentation
                if random.random() < 0.1:
                    _action_mask_aug_per = 0.0
                else:
                    _action_mask_aug_per = random.uniform(0.0, self.action_mask_aug_per)
                mask_len = int(len(_action_txt) * _action_mask_aug_per)
                mask_indices = random.sample(range(len(_action_txt)), mask_len)
                mask_indices = [
                    x + sysuser_len for x in mask_indices
                ]  # add sysuser_len to the mask indices to get the correct indices of these tokens
                labels[
                    i, mask_indices
                ] = -100  # these elements will not be used for loss calculation
                model_inputs["input_ids"][
                    i, mask_indices
                ] = 30  # replace the input ids with '?' token id

            labels[labels == 151643] = -100

            outputs = self.model(**model_inputs)
            logits = outputs.logits  # (batch_size, seq_len, vocab_size)

            # copied from modeling_qwen2_5_vl.py to compute the loss
            # Upcast to float if we need to compute the loss to avoid potential precision issues
            logits = logits.float()
            # Shift so that tokens < n predict n
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            # Flatten the tokens
            shift_logits = shift_logits.view(-1, shift_logits.size(-1))
            shift_labels = shift_labels.view(-1)
            # Enable model parallelism
            shift_labels = shift_labels.to(shift_logits.device)
            loss = self.loss_fn(shift_logits, shift_labels)

            return {"loss": loss}

        if get_action:
            sample_args = {}
            if generate_temperature > 0:
                sample_args["temperature"] = generate_temperature
            else:
                sample_args[
                    "do_sample"
                ] = False  # greedy search, this makes the generation deterministic

            if get_one_step_action:
                # we calculate the max number of tokens to generate for one step of action
                # +1 is for the space token
                max_new_tokens = self.act_dim * (len(str(self.num_bins_actions)) + 1)
                if last_action_txt != "":
                    last_action_txt_ids = self.processor.tokenizer(
                        last_action_txt, return_tensors="pt"
                    )["input_ids"].to(model_inputs["input_ids"].device)
                    model_inputs["input_ids"] = torch.cat(
                        [model_inputs["input_ids"], last_action_txt_ids], dim=1
                    )
                    model_inputs["attention_mask"] = torch.cat(
                        [
                            model_inputs["attention_mask"],
                            torch.ones_like(last_action_txt_ids),
                        ],
                        dim=1,
                    )

            # token id to text mapping
            # 220 is space
            # 15 to 24 are 0 to 9
            # 151645 is the end of text token
            generated_ids = self.model.generate(
                **model_inputs,
                logits_processor=[self.logits_processor],
                max_new_tokens=(
                    max_new_tokens if get_one_step_action else self.num_tokens
                ),
                **sample_args,
            )

            input_ids = model_inputs["input_ids"]
            generated_ids_trimmed = [
                out_ids[len(in_ids) :]
                for in_ids, out_ids in zip(input_ids, generated_ids)
            ]
            generated_action_txt = self.processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )

            if get_one_step_action:
                # TODO: Only supports batch size 1
                generated_action_txt = [last_action_txt + generated_action_txt[0]]

            out_ori_act = self.get_action_from_text_action(
                generated_action_txt, instruction=instr
            )

            if get_one_step_action:
                out_ori_act = out_ori_act[:, -1:]

            return {
                "gt_action_text": action_txt,
                "pred_action_txt": generated_action_txt,
                "gt_out_ori_act": out_ori_act,
                "out_ori_act": out_ori_act,
            }

    def save_pretrained(self, path):
        self.model.save_pretrained(path)
        self.processor.save_pretrained(path)

    def from_pretrained(self, path, is_trainable=True):
        _device = next(self.parameters()).device
        # This way works
        del self.model
        torch.cuda.empty_cache()
        gc.collect()

        if self.use_lora:
            from peft import PeftModel

            base_model = self.load_qwen_model(
                qwen_model_id=self.qwen_model_id,
                use_lora=self.use_lora,
                use_qlora=self.use_qlora,
                lora_config="",  # None regarless of lora config as lora is added later using PeftModel
                lora_rank=self.lora_rank,  # Doesn't matter here as lora_config is None
                use_flash_attention_2=self.use_flash_attention_2,
                attention_dropout=self.attention_dropout,
            )

            self.model = PeftModel.from_pretrained(
                base_model,
                path,
                is_trainable=is_trainable,
            )
            print("Loading Qwen2.5 PEFT model from", path)
        else:
            extra_kwargs = {}
            if self.use_flash_attention_2:
                extra_kwargs["attn_implementation"] = "flash_attention_2"
            if self.attention_dropout > 0.0:
                config = AutoConfig.from_pretrained(path)
                config.attention_dropout = self.attention_dropout
                extra_kwargs["config"] = config
            self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                path,
                # device_map={"": "cuda:0"},
                torch_dtype=torch.bfloat16,
                **extra_kwargs,
            )
            print("Loading Qwen2.5 full model from", path)

        if self.use_flash_attention_2:
            self.processor = Qwen2_5_VLProcessor.from_pretrained(
                path,
                min_pixels=self.min_pixel,
                max_pixels=self.max_pixel,
                padding_side="left",
            )
        else:
            self.processor = Qwen2_5_VLProcessor.from_pretrained(
                path,
                min_pixels=self.min_pixel,
                max_pixels=self.max_pixel,
            )

        print("Loading Qwen2.5 processor from", path)

        VLA0_QwenActor.to(self, _device)

    def to(self, device):
        super().to(device)
        # if device is interger like 0 or "0", convert to cuda:0
        if isinstance(device, int) or (isinstance(device, str) and device.isnumeric()):
            device = f"cuda:{device}"
        if hasattr(self, "renderer"):
            self.renderer.renderer.device = device
            self.renderer.cameras.to(device)

    def tile_images(self, images):
        """
        Tile a images into a single image
        :param images: Tensor of shape (bs, H, W, 3) or list of tensors of shape (H, W, 3)
        :return: Tensor of shape (bs, H, W, 3)
        """
        for img in images:
            assert len(img.shape) == 3, f"img.shape: {img.shape}"
            assert img.shape[2] == 3, f"img.shape: {img.shape}"

        widths, heights = zip(*(im.shape[:-1] for im in images))
        total_width = sum(widths)
        max_height = max(heights)
        dst = torch.zeros((max_height, total_width, 3), device=images[0].device)
        current_x = 0
        for i, img in enumerate(images):
            dst[: img.shape[0], current_x : current_x + img.shape[1], :] = img
            current_x += img.shape[1]
        return dst

In [ ]:
class Fresh_QwenActor(nn.Module):
    def __init__(
        self,
        qwen_model_id,
        action_type,
        original_action_dim,
        horizon,
        history,
        use_lora=True,
        use_qlora=True,
        num_cam=1,
        lora_config="",
        lora_rank=8,
        rgb_input=False,
        rgb_img_size=(84, 84),
        add_vision_id=False,
        tiled_rgb_imgs=False,
        num_bins_actions=1000,
        use_flash_attention_2=False,
        system_message_version=1,
        action_mask_aug=0,
        action_mask_aug_per=0.1,
        attention_dropout=0.0,
    ):
        """
        :param qwen_model_id: str, the id of the qwen model to use
        :param action_type: str, the type of action to use, either ORIGINAL or EE
        :param original_action_dim: int, the dimension of the original action
        :param horizon: int, the horizon of the action
        :param history: int, the history of the action
        :param use_qlora: bool, whether to use qlora for parameter efficient fine-tuning
        :param num_cam: int, the number of cameras for rgb input
        :param lora_config: str, the lora configuration to use, either empty string or "default"
        :param lora_rank: int, the rank of the lora to use, only used if lora_config is "default"
        :param rgb_input: bool, whether to use rgb image input
        :param rgb_img_size: tuple, the size of the rgb image input (height, width)
        :param add_vision_id: bool, whether to add vision id to the input for qwen2.5
        :param tiled_rgb_imgs: bool, whether to tile the rgb images into a single image instead feeding them separately
        :param num_bins_actions: int, the number of bins in which each action dimension is discretized
        :param use_flash_attention_2: bool, whether to use flash attention 2 for faster training and inference
        :param attention_dropout: float, the dropout rate for the attention layer in the qwen model. Only tested when use_lora is False.
        """
        super().__init__()

        # current assumptions
        if use_qlora:
            assert use_lora, "use_lora must be True if use_qlora is True"
        if attention_dropout > 0.0:
            assert (
                not use_lora
            ), "attention_dropout is only supported when use_lora is False"
        assert lora_config in ["", "default"]
        if history > 1 or num_cam > 1:
            assert (
                add_vision_id
            ), "add_vision_id must be True if history > 1 or num_cam > 1"

        # assert not use_flash_attention_2, "use_flash_attention_2 is not supported yet, it requires tokenizer.pad=left which we have not fully implemented/understood"

        # for Qwen model, we need to load the parameters before DDP
        # in case we want to load the model from a checkpoint
        self.load_param_before_ddp = True

        self.qwen_model_id = qwen_model_id
        self.action_type = action_type
        self.original_action_dim = original_action_dim
        self.horizon = horizon
        self.history = history
        self.use_lora = use_lora
        self.use_qlora = use_qlora
        self.num_cam = num_cam
        self.lora_config = lora_config
        self.lora_rank = lora_rank
        self.rgb_input = rgb_input
        self.rgb_img_size = rgb_img_size
        self.add_vision_id = add_vision_id
        self.tiled_rgb_imgs = tiled_rgb_imgs
        self.num_bins_actions = num_bins_actions
        self.use_flash_attention_2 = use_flash_attention_2
        self.action_mask_aug_per = action_mask_aug_per
        self.attention_dropout = attention_dropout

        self.model = self.load_qwen_model(
            qwen_model_id=self.qwen_model_id,
            use_lora=self.use_lora,
            use_qlora=self.use_qlora,
            lora_config=self.lora_config,
            lora_rank=self.lora_rank,
            use_flash_attention_2=self.use_flash_attention_2,
            attention_dropout=self.attention_dropout,
        )

        # Enable gradient checkpointing if requested
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

        self.min_pixel = self.max_pixel = self.rgb_img_size[0] * self.rgb_img_size[1]
        if self.rgb_input and self.tiled_rgb_imgs:
            self.min_pixel *= self.history * self.num_cam
            self.max_pixel *= self.history * self.num_cam

        self.processor = self.load_qwen_model_processor(
            qwen_model_id=qwen_model_id,
            min_pixel=self.min_pixel,
            max_pixel=self.max_pixel,
            padding_side="left" if use_flash_attention_2 else None,
        )
        self.logits_processor = NumberSpaceOnlyProcessor(self.processor.tokenizer)

        print(
            "WARNING: Using hardcoded dataset stats for DP3. This should be replaced with loading from a file."
        )

        if action_type == C.ORIGINAL:
            self.act_dim = original_action_dim
        elif action_type == C.EE:
            self.act_dim = 7
        else:
            assert False

        # ── CHANGED ──────────────────────────────────────────────────────────
        # Ask the model to write a Python function instead of a flat number
        # sequence.  The function `get_action(step_id)` accepts an integer in
        # [0, horizon-1] and returns a list of `act_dim` integers in
        # [0, num_bins_actions].  The model is encouraged to express the
        # trajectory analytically (e.g. sinusoidal, polynomial, exponential
        # easing) using numpy, math, or torch rather than a plain lookup table.
        self.system_message = (
            f"Analyze the input image and write a Python function named "
            f"'get_action' that takes a single integer argument 'step_id' "
            f"(ranging from 0 to {self.horizon - 1}) and returns a list of "
            f"{self.act_dim} integers (each clamped to [0, {self.num_bins_actions}]), "
            f"representing the predicted robot actions for that future timestep. "
            f"Express the trajectory analytically — use numpy, math, or torch to "
            f"compute smooth, physically plausible motion (e.g. sinusoidal curves, "
            f"polynomial easing, exponential approach, or spline-like interpolation). "
            f"Avoid plain static lookup tables. "
            f"Import any modules you need at the top of the function body. "
            f"Output only the Python function definition. Nothing else."
        )
        # ── END CHANGED ───────────────────────────────────────────────────────

        # todo: need better way to determine it
        self.num_tokens = 1024
        self.original_dataset_stats = None  # original dataset stats has the same format as the one provided by the dataset
        self.dataset_stats = (
            None  # dataset stats is in the format specific to the model
        )

        self._sysuser_len = None

        self.cache_sysuser_len = False

        self._forward_count = 0

    def set_dataset_stats(self, dataset_stats):
        """
        Set the dataset stats for the model
        :param dataset_stats: dict, the dataset stats
        """
        if dataset_stats == {}:
            warnings.warn(
                "Dataset stats is empty likely because the system does not have the data used to compute the stats. Ignore this is you are loading a pretrained model."
            )
            return

        self.original_dataset_stats = dataset_stats
        if self.action_type == C.ORIGINAL:
            self.dataset_stats = dataset_stats["out_ori_act"]
        else:
            raise NotImplementedError(f"Action type {self.action_type} not implemented")

    @staticmethod
    def load_qwen_model(
        qwen_model_id,
        use_lora,
        use_qlora,
        lora_config,
        lora_rank,
        use_flash_attention_2,
        attention_dropout,
    ):
        if lora_config == "":
            lora_config = None
        elif lora_config == "default":
            if use_lora:
                from peft import LoraConfig, get_peft_model

                lora_config = LoraConfig(
                    lora_alpha=16,
                    lora_dropout=0.05,
                    r=lora_rank,
                    bias="none",
                    target_modules=["q_proj", "v_proj"],
                    task_type="CAUSAL_LM",
                )
        else:
            raise ValueError(f"Invalid lora_config: {lora_config}")

        # Qwen Init
        bnb_config = None
        if use_lora and use_qlora:
            from transformers import BitsAndBytesConfig

            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_type=torch.bfloat16,
            )

        extra_kwargs = {}
        if use_flash_attention_2:
            extra_kwargs["attn_implementation"] = "flash_attention_2"

        if attention_dropout > 0.0:
            config = AutoConfig.from_pretrained(qwen_model_id)
            config.attention_dropout = attention_dropout
            extra_kwargs["config"] = config

        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            qwen_model_id,
            # device_map={"": "cuda:0"},  # Use the explicit map
            quantization_config=bnb_config,
            torch_dtype=torch.bfloat16,
            # local_files_only=True,
            **extra_kwargs,
        )

        if use_lora and (lora_config is not None):
            model = get_peft_model(model, lora_config)

        return model

    @staticmethod
    def load_qwen_model_processor(
        qwen_model_id,
        min_pixel,
        max_pixel,
        padding_side,
    ):
        if padding_side is not None:
            processor = Qwen2_5_VLProcessor.from_pretrained(
                qwen_model_id,
                min_pixels=min_pixel,
                max_pixels=max_pixel,
                padding_side=padding_side,
                # local_files_only=True,
            )
        else:
            processor = Qwen2_5_VLProcessor.from_pretrained(
                qwen_model_id,
                min_pixels=min_pixel,
                max_pixels=max_pixel,
                # local_files_only=True,
            )

        return processor

    def get_min_max_act(self, instruction):
        """
        Get the min and max action for the instruction.
        :param instruction: str, the instruction for the current episode. This is needed for libero bounds 99% as the action space is different for different instructions.
        :return: torch.Tensor, the min and max action.
        """
        assert instruction is not None, "instruction is needed for libero bounds 99%"
        min_act = []
        max_act = []
        for _instruction in instruction:
            _suite = self.instruction_to_suite[_instruction]
            min_act.append(torch.tensor(self.dataset_stats[_suite]["min"]))
            max_act.append(torch.tensor(self.dataset_stats[_suite]["max"]))
        min_act = torch.stack(min_act, dim=0)
        max_act = torch.stack(max_act, dim=0)
        return min_act, max_act

    def get_text_action(self, actions, instruction=None):
        """
        Get the text action from the actions.
        :param actions: torch.Tensor, the actions to convert to text, shape (bs, horizon, act_dim).
        :param instruction: str, the instruction for the current episode.
        :return: List[str], one Python function string per sample in the batch.
        """
        # ── CHANGED ──────────────────────────────────────────────────────────
        # Previously this produced a flat space-separated string of integers.
        # Now it produces a Python function `get_action(step_id)` whose body
        # contains the full action table as a literal list-of-lists so the
        # ground-truth target matches the new generation format.
        if (
            (not hasattr(self, "_min_act"))
            or (not hasattr(self, "_max_act"))
            or (self._min_act.device != actions.device)
            or (self._max_act.device != actions.device)
        ):
            min_act = torch.tensor(self.dataset_stats["min"], device=actions.device)
            max_act = torch.tensor(self.dataset_stats["max"], device=actions.device)
            self._min_act = min_act
            self._max_act = max_act
        else:
            min_act = self._min_act
            max_act = self._max_act

        assert torch.all(min_act <= actions) and torch.all(
            actions <= max_act
        ), f"Action is out of range: {actions}"

        # Normalise → bin indices, shape (bs, horizon, act_dim)
        actions = (actions - min_act) / (max_act - min_act)
        actions *= self.num_bins_actions
        actions = torch.round(actions).long()  # (bs, horizon, act_dim)

        action_txt = []
        for sample_actions in actions:
            # sample_actions: (horizon, act_dim) integer tensor
            table = sample_actions.tolist()  # list[list[int]]
            fn = (
                "def get_action(step_id):\n"
                f"    actions = {table}\n"
                "    return actions[step_id]"
            )
            action_txt.append(fn)

        return action_txt
        # ── END CHANGED ───────────────────────────────────────────────────────

    def get_action_from_text_action(self, action_txt, instruction=None):
        """
        Get the action from the text action.
        :param action_txt: List[str], one Python function string per sample.
        :param instruction: str, the instruction for the current episode.
        :return: torch.Tensor of shape (bs, horizon, act_dim).
        """
        # ── CHANGED ──────────────────────────────────────────────────────────
        # Previously this parsed a flat space-separated integer string.
        # Now it exec-utes the generated `get_action(step_id)` function and
        # calls it for every step_id in [0, horizon-1], then de-normalises.
        bs = len(action_txt)
        min_act = torch.tensor(self.dataset_stats["min"])
        max_act = torch.tensor(self.dataset_stats["max"])

        results = []
        for txt in action_txt:
            try:
                fn_start = txt.find("def get_action")

                if fn_start != -1:
                    # ── new format: execute the generated Python function ──
                    # Strip markdown code fences the model may emit (```python ... ```)
                    txt_clean = txt
                    for fence in ("```python", "```"):
                        txt_clean = txt_clean.replace(fence, "")
                    fn_start = txt_clean.find("def get_action")
                    fn_src = txt_clean[fn_start:]
                    # Drop any top-level statements after the function body
                    # (e.g. `action = get_action(3)` / `print(...)` the model sometimes appends)
                    import ast as _ast
                    try:
                        tree = _ast.parse(fn_src)
                        # Keep only the function def node
                        fn_nodes = [n for n in tree.body if isinstance(n, _ast.FunctionDef)]
                        if fn_nodes:
                            fn_src = _ast.get_source_segment(fn_src, fn_nodes[0]) or fn_src
                    except Exception:
                        pass  # fall through to exec; it will raise a clearer error if needed
                    local_ns: dict = {}
                    exec(fn_src, {"__builtins__": __builtins__}, local_ns)  # noqa: S102
                    get_action_fn = local_ns["get_action"]

                    step_actions = []
                    for step_id in range(self.horizon):
                        step = get_action_fn(step_id)
                        step = list(step)[: self.act_dim]
                        while len(step) < self.act_dim:
                            step.append(0)
                        step_actions.append([int(v) for v in step])

                    action = torch.tensor(step_actions, dtype=torch.float32)  # (horizon, act_dim)

                else:
                    # ── legacy fallback: flat space-separated integer string ──
                    tokens = txt.strip().split()
                    flat = [int(x) for x in tokens]
                    # make length divisible by act_dim
                    remainder = len(flat) % self.act_dim
                    if remainder != 0:
                        flat = flat[: len(flat) - remainder]
                    action = torch.tensor(flat, dtype=torch.float32).reshape(-1, self.act_dim)
                    # pad / trim to exactly horizon steps
                    if action.shape[0] < self.horizon:
                        action = torch.cat(
                            [action, action[-1:].repeat(self.horizon - action.shape[0], 1)],
                            dim=0,
                        )
                    action = action[: self.horizon]

                # de-normalise (shared by both paths)
                action = ((action / self.num_bins_actions) * (max_act - min_act)) + min_act

            except Exception as e:
                print(f"Error in parsing action function: {e}")
                print(txt)
                action = ((min_act + max_act) / 2).unsqueeze(0).repeat(self.horizon, 1)

            results.append(action)

        return torch.stack(results, dim=0)  # (bs, horizon, act_dim)
        # ── END CHANGED ───────────────────────────────────────────────────────

    def check_inputs(
        self,
        pc,
        rgb_pc,
        instr,
        rgb,
        ori_act,
        ee_pos,
        ee_rot,
        ee_gri,
        out_ee_pos,
        out_ee_rot,
        out_ee_gri,
        out_ori_act,
        get_loss,
        get_action,
        get_one_step_action,
        last_action_txt,
    ):
        assert (
            self.dataset_stats is not None
        ), "dataset_stats must be set before calling forward"
        assert isinstance(instr, list)

        assert not (get_loss and get_action)
        assert get_loss or get_action
        if get_one_step_action:
            assert get_action
            assert isinstance(last_action_txt, str)
            assert len(instr) == 1, "one_step_action is only supported for batch size 1"
        if self.rgb_input:
            assert rgb is not None
            assert rgb.shape[1:] == (
                self.history,
                self.num_cam,
                *self.rgb_img_size,
                3,
            ), f"rgb.shape: {rgb.shape}, self.history: {self.history}, self.num_cam: {self.num_cam}, self.rgb_img_size: {self.rgb_img_size}"
            # some room for numerical errors
            # 0, 2, 255 are valid values
            assert (rgb.min() >= -1e-2) and (
                1.99 <= rgb.max() <= 255.01
            ), f"rgb.min(): {rgb.min()}, rgb.max(): {rgb.max()}"
        else:
            assert rgb is None

        assert pc is None
        assert rgb_pc is None

    def get_imgs(
        self,
        bs,
        pc,
        rgb_pc,
        rgb,
    ):
        """
        Get the images for the given inputs
        :param bs: int, the batch size
        :param pc: torch.Tensor, the point cloud
        :param rgb_pc: torch.Tensor, the rgb point cloud
        :param rgb: torch.Tensor, the rgb image
        :return: list of list of PIL images
        """
        imgs = [[] for _ in range(bs)]

        if self.rgb_input:
            for i, _rgb in enumerate(rgb):
                _imgs = []
                for j in range(self.history):
                    for k in range(self.num_cam):
                        _imgs.append(_rgb[j][k])
                if self.tiled_rgb_imgs:
                    _imgs = [self.tile_images(_imgs)]
                _imgs = [
                    Image.fromarray(x.cpu().numpy().astype(np.uint8)) for x in _imgs
                ]
                imgs[i].extend(_imgs)

        return imgs

    def get_qwen_inputs(
        self,
        bs: int,
        imgs: List[List[PIL.Image.Image]],
        instr: List[str],
        action_txt: List[str],
        drop_assistant: bool = False,
        add_generation_prompt: bool = False,
    ):
        """
        Get the Qwen inputs for the given inputs
        :param bs: int, the batch size
        :param imgs: list of list of PIL images
        :param instr: list of strings
        :param action_txt: list of strings
        :param drop_assistant: bool, whether to drop the assistant portion.
        :param add_generation_prompt: bool, whether to add the generation prompt assistant\n.
        """

        examples = [
            format_data(
                system_message=self.system_message,
                image=imgs[i],
                instr=instr[i],
                action_txt=action_txt[i],
            )
            for i in range(bs)
        ]
        if drop_assistant:
            # drop the assistant portion so the model must generate it
            examples = [e[:2] for e in examples]

        texts = [
            self.processor.apply_chat_template(
                example,
                tokenize=False,
                add_generation_prompt=add_generation_prompt,
                add_vision_id=self.add_vision_id,
            )
            # when add_generation_prompt is True, it will add the prompt
            # `assistant\n` to the end of the input text
            for example in examples
        ]
        # [0] in process_vision_info is for image input, [1] is for video input
        image_inputs = [process_vision_info(example)[0] for example in examples]

        model_inputs = self.processor(
            text=texts, images=image_inputs, return_tensors="pt", padding=True
        )
        for key in model_inputs:
            model_inputs[key] = model_inputs[key].to(
                next(self.model.parameters()).device
            )
        return model_inputs, examples

    def forward(
        self,
        pc=None,
        rgb_pc=None,
        instr=None,
        rgb=None,
        ori_act=None,
        ee_pos=None,
        ee_rot=None,
        ee_gri=None,
        out_ee_pos=None,
        out_ee_rot=None,
        out_ee_gri=None,
        out_ori_act=None,
        get_loss=True,
        get_action=False,
        generate_temperature=0.1,
        get_one_step_action=False,
        last_action_txt="",
    ):
        """
        Forward pass for the Qwen model

        Parameters
        ----------
        pc : optional
            Point cloud data
        rgb_pc : optional
            RGB point cloud data
        instr : optional
            Instructions
        rgb : optional
            RGB images
        ori_act : optional
            Original actions
        ee_pos : optional
            End effector position
        ee_rot : optional
            End effector rotation
        ee_gri : optional
            End effector gripper state
        out_ee_pos : optional
            Output end effector position
        out_ee_rot : optional
            Output end effector rotation
        out_ee_gri : optional
            Output end effector gripper state
        out_ori_act : optional
            Output original actions
        get_loss : bool, default=True
            Whether to calculate and return loss
        get_action : bool, default=False
            Whether to get actions
        generate_temperature : float, default=0.1
            Temperature for generation
        get_one_step_action : bool, default=False
            Whether to run the model forward for only one step of action at a time. If True,
            we complete the last_action_txt to next action string sufficient to decode the
            action for one next step. If False, we complete the last_action_txt to next
            action string sufficient to decode the action for all the next steps.
        last_action_txt : str, default=""
            The last action text to complete to get the next action text. This is only
            used when get_one_step_action is True.

        Returns
        -------
        dict or tuple
            Model outputs (should specify exact return structure)

        Raises
        ------
        Exception
            Any exceptions that can be raised (if applicable)
        """
        self.check_inputs(
            pc=pc,
            rgb_pc=rgb_pc,
            instr=instr,
            rgb=rgb,
            ori_act=ori_act,
            ee_pos=ee_pos,
            ee_rot=ee_rot,
            ee_gri=ee_gri,
            out_ee_pos=out_ee_pos,
            out_ee_rot=out_ee_rot,
            out_ee_gri=out_ee_gri,
            out_ori_act=out_ori_act,
            get_loss=get_loss,
            get_action=get_action,
            get_one_step_action=get_one_step_action,
            last_action_txt=last_action_txt,
        )

        bs = len(instr)

        # imgs is list of list of PIL images
        imgs = self.get_imgs(
            bs=bs,
            pc=pc,
            rgb_pc=rgb_pc,
            rgb=rgb,
        )

        # TODO: implement this for ee action space
        if out_ori_act is None:
            assert not get_loss
            action_txt = [[]] * bs
        else:
            action_txt = self.get_text_action(out_ori_act, instruction=instr)

        model_inputs, examples = self.get_qwen_inputs(
            bs=bs,
            imgs=imgs,
            instr=instr,
            action_txt=action_txt,
            drop_assistant=get_action,  # when getting action, we drop the assistant portion
            add_generation_prompt=get_action,  # when getting action, we add the generation prompt assistant\n so that the model need not generate it
        )

        if get_loss:
            labels = model_inputs["input_ids"].clone()
            # mask system message and image token IDs in the labels
            for i, example in enumerate(examples):
                if (self._sysuser_len is None) or (not self.cache_sysuser_len):
                    sysuser_conv = example[:-1]
                    sysuser_text = self.processor.apply_chat_template(
                        sysuser_conv, tokenize=False, add_vision_id=self.add_vision_id
                    )
                    sysuser_img, _ = process_vision_info(sysuser_conv)

                    sysuser_inputs = self.processor(
                        text=[sysuser_text],
                        images=[sysuser_img],
                        return_tensors="pt",
                        padding=True,
                    )

                    sysuser_len = sysuser_inputs["input_ids"].shape[1]
                    sysuser_len += 3  # to mask out `assistant\n`
                    self._sysuser_len = sysuser_len
                else:
                    sysuser_len = self._sysuser_len
                # TIP: to decode the input use:
                # when padding is right: self.processor.decode(model_inputs["input_ids"][0][0:sysuser_len])
                # when padding is left: self.processor.decode(model_inputs["input_ids"][0][num_pad_tokens: num_pad_tokens + sysuser_len])
                if self.processor.tokenizer.padding_side == "right":
                    labels[i, :sysuser_len] = -100
                elif self.processor.tokenizer.padding_side == "left":
                    num_pad_tokens = sum(labels[i] == 151643).item()
                    labels[i, num_pad_tokens : num_pad_tokens + sysuser_len] = -100
                else:
                    raise ValueError(
                        f"Unknown padding side: {self.processor.tokenizer.padding_side}"
                    )

                assert (
                    not self.processor.tokenizer.padding_side == "left"
                ), "current implementation only supports right padding"
                # for debugging, compare
                # self.processor.decode(model_inputs["input_ids"][i][model_inputs["attention_mask"][i] == 1])
                # with self.processor.decode(model_inputs["input_ids"][i])
                _action_txt = action_txt[i]
                # 10% of sample has no augmentation
                if random.random() < 0.1:
                    _action_mask_aug_per = 0.0
                else:
                    _action_mask_aug_per = random.uniform(0.0, self.action_mask_aug_per)
                mask_len = int(len(_action_txt) * _action_mask_aug_per)
                mask_indices = random.sample(range(len(_action_txt)), mask_len)
                mask_indices = [
                    x + sysuser_len for x in mask_indices
                ]  # add sysuser_len to the mask indices to get the correct indices of these tokens
                labels[
                    i, mask_indices
                ] = -100  # these elements will not be used for loss calculation
                model_inputs["input_ids"][
                    i, mask_indices
                ] = 30  # replace the input ids with '?' token id

            labels[labels == 151643] = -100

            outputs = self.model(**model_inputs)
            logits = outputs.logits  # (batch_size, seq_len, vocab_size)

            # copied from modeling_qwen2_5_vl.py to compute the loss
            # Upcast to float if we need to compute the loss to avoid potential precision issues
            logits = logits.float()
            # Shift so that tokens < n predict n
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            # Flatten the tokens
            shift_logits = shift_logits.view(-1, shift_logits.size(-1))
            shift_labels = shift_labels.view(-1)
            # Enable model parallelism
            shift_labels = shift_labels.to(shift_logits.device)
            loss = self.loss_fn(shift_logits, shift_labels)

            return {"loss": loss}

        if get_action:
            sample_args = {}
            if generate_temperature > 0:
                sample_args["temperature"] = generate_temperature
            else:
                sample_args[
                    "do_sample"
                ] = False  # greedy search, this makes the generation deterministic

            if get_one_step_action:
                # we calculate the max number of tokens to generate for one step of action
                # +1 is for the space token
                max_new_tokens = self.act_dim * (len(str(self.num_bins_actions)) + 1)
                if last_action_txt != "":
                    last_action_txt_ids = self.processor.tokenizer(
                        last_action_txt, return_tensors="pt"
                    )["input_ids"].to(model_inputs["input_ids"].device)
                    model_inputs["input_ids"] = torch.cat(
                        [model_inputs["input_ids"], last_action_txt_ids], dim=1
                    )
                    model_inputs["attention_mask"] = torch.cat(
                        [
                            model_inputs["attention_mask"],
                            torch.ones_like(last_action_txt_ids),
                        ],
                        dim=1,
                    )

            # token id to text mapping
            # 220 is space
            # 15 to 24 are 0 to 9
            # 151645 is the end of text token
            # ── CHANGED ──────────────────────────────────────────────────────
            # logits_processor is removed: the model must now emit arbitrary
            # Python source code, so constraining to digits/spaces would
            # prevent it from generating keywords, brackets, etc.
            generated_ids = self.model.generate(
                **model_inputs,
                max_new_tokens=(
                    max_new_tokens if get_one_step_action else self.num_tokens
                ),
                **sample_args,
            )
            # ── END CHANGED ───────────────────────────────────────────────────

            input_ids = model_inputs["input_ids"]
            generated_ids_trimmed = [
                out_ids[len(in_ids) :]
                for in_ids, out_ids in zip(input_ids, generated_ids)
            ]
            generated_action_txt = self.processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )

            if get_one_step_action:
                # TODO: Only supports batch size 1
                generated_action_txt = [last_action_txt + generated_action_txt[0]]

            # ── CHANGED ──────────────────────────────────────────────────────
            self._forward_count += 1
            if self._forward_count % 50 == 0:
                print(
                    f"\n{'='*60}\n"
                    f"[Fresh_QwenActor] Generated function @ forward pass {self._forward_count} "
                    f"(batch sample 0):\n"
                    f"{'─'*60}\n"
                    f"{generated_action_txt[0]}\n"
                    f"{'='*60}\n"
                )
            # ── END CHANGED ───────────────────────────────────────────────────

            out_ori_act = self.get_action_from_text_action(
                generated_action_txt, instruction=instr
            )

            if get_one_step_action:
                out_ori_act = out_ori_act[:, -1:]

            return {
                "gt_action_text": action_txt,
                "pred_action_txt": generated_action_txt,
                "gt_out_ori_act": out_ori_act,
                "out_ori_act": out_ori_act,
            }

    def save_pretrained(self, path):
        self.model.save_pretrained(path)
        self.processor.save_pretrained(path)

    def from_pretrained(self, path, is_trainable=True):
        print(f"[Fresh_QwenActor] Skipping finetuned checkpoint '{path}' — using fresh pretrained weights from '{self.qwen_model_id}'.")
        return

        _device = next(self.parameters()).device
        # This way works
        del self.model
        torch.cuda.empty_cache()
        gc.collect()

        if self.use_lora:
            from peft import PeftModel

            base_model = self.load_qwen_model(
                qwen_model_id=self.qwen_model_id,
                use_lora=self.use_lora,
                use_qlora=self.use_qlora,
                lora_config="",  # None regarless of lora config as lora is added later using PeftModel
                lora_rank=self.lora_rank,  # Doesn't matter here as lora_config is None
                use_flash_attention_2=self.use_flash_attention_2,
                attention_dropout=self.attention_dropout,
            )

            self.model = PeftModel.from_pretrained(
                base_model,
                path,
                is_trainable=is_trainable,
            )
            print("Loading Qwen2.5 PEFT model from", path)
        else:
            extra_kwargs = {}
            if self.use_flash_attention_2:
                extra_kwargs["attn_implementation"] = "flash_attention_2"
            if self.attention_dropout > 0.0:
                config = AutoConfig.from_pretrained(path)
                config.attention_dropout = self.attention_dropout
                extra_kwargs["config"] = config
            self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                path,
                # device_map={"": "cuda:0"},
                torch_dtype=torch.bfloat16,
                **extra_kwargs,
            )
            print("Loading Qwen2.5 full model from", path)

        if self.use_flash_attention_2:
            self.processor = Qwen2_5_VLProcessor.from_pretrained(
                path,
                min_pixels=self.min_pixel,
                max_pixels=self.max_pixel,
                padding_side="left",
            )
        else:
            self.processor = Qwen2_5_VLProcessor.from_pretrained(
                path,
                min_pixels=self.min_pixel,
                max_pixels=self.max_pixel,
            )

        print("Loading Qwen2.5 processor from", path)

        Fresh_QwenActor.to(self, _device)

    def to(self, device):
        super().to(device)
        # if device is interger like 0 or "0", convert to cuda:0
        if isinstance(device, int) or (isinstance(device, str) and device.isnumeric()):
            device = f"cuda:{device}"
        if hasattr(self, "renderer"):
            self.renderer.renderer.device = device
            self.renderer.cameras.to(device)

    def tile_images(self, images):
        """
        Tile a images into a single image
        :param images: Tensor of shape (bs, H, W, 3) or list of tensors of shape (H, W, 3)
        :return: Tensor of shape (bs, H, W, 3)
        """
        for img in images:
            assert len(img.shape) == 3, f"img.shape: {img.shape}"
            assert img.shape[2] == 3, f"img.shape: {img.shape}"

        widths, heights = zip(*(im.shape[:-1] for im in images))
        total_width = sum(widths)
        max_height = max(heights)
        dst = torch.zeros((max_height, total_width, 3), device=images[0].device)
        current_x = 0
        for i, img in enumerate(images):
            dst[: img.shape[0], current_x : current_x + img.shape[1], :] = img
            current_x += img.shape[1]
        return dst

In [ ]:
import gc
import random
import warnings
from typing import List

import numpy as np
import PIL
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info
from torch import nn
from transformers import AutoConfig, AutoProcessor, LogitsProcessor
from transformers.models.qwen3_vl.modeling_qwen3_vl import \
    Qwen3VLForConditionalGeneration

import rv_train.constants as C

class Coding_QwenActor(nn.Module):
    def __init__(
        self,
        qwen_model_id,
        action_type,
        original_action_dim,
        horizon,
        history,
        use_lora=True,
        use_qlora=True,
        num_cam=1,
        lora_config="",
        lora_rank=8,
        rgb_input=False,
        rgb_img_size=(84, 84),
        add_vision_id=False,
        tiled_rgb_imgs=False,
        num_bins_actions=1000,
        use_flash_attention_2=False,
        system_message_version=1,
        action_mask_aug=0,
        action_mask_aug_per=0.1,
        attention_dropout=0.0,
    ):
        """
        :param qwen_model_id: str, the id of the qwen model to use
        :param action_type: str, the type of action to use, either ORIGINAL or EE
        :param original_action_dim: int, the dimension of the original action
        :param horizon: int, the horizon of the action
        :param history: int, the history of the action
        :param use_qlora: bool, whether to use qlora for parameter efficient fine-tuning
        :param num_cam: int, the number of cameras for rgb input
        :param lora_config: str, the lora configuration to use, either empty string or "default"
        :param lora_rank: int, the rank of the lora to use, only used if lora_config is "default"
        :param rgb_input: bool, whether to use rgb image input
        :param rgb_img_size: tuple, the size of the rgb image input (height, width)
        :param add_vision_id: bool, whether to add vision id to the input for qwen2.5
        :param tiled_rgb_imgs: bool, whether to tile the rgb images into a single image instead feeding them separately
        :param num_bins_actions: int, the number of bins in which each action dimension is discretized
        :param use_flash_attention_2: bool, whether to use flash attention 2 for faster training and inference
        :param attention_dropout: float, the dropout rate for the attention layer in the qwen model. Only tested when use_lora is False.
        """
        super().__init__()
        qwen_model_id="Qwen/Qwen3-VL-4B-Instruct"

        # current assumptions
        if use_qlora:
            assert use_lora, "use_lora must be True if use_qlora is True"
        if attention_dropout > 0.0:
            assert (
                not use_lora
            ), "attention_dropout is only supported when use_lora is False"
        assert lora_config in ["", "default"]
        if history > 1 or num_cam > 1:
            assert (
                add_vision_id
            ), "add_vision_id must be True if history > 1 or num_cam > 1"

        # assert not use_flash_attention_2, "use_flash_attention_2 is not supported yet, it requires tokenizer.pad=left which we have not fully implemented/understood"

        # for Qwen model, we need to load the parameters before DDP
        # in case we want to load the model from a checkpoint
        self.load_param_before_ddp = True

        self.qwen_model_id = qwen_model_id
        self.action_type = action_type
        self.original_action_dim = original_action_dim
        self.horizon = horizon
        self.history = history
        self.use_lora = use_lora
        self.use_qlora = use_qlora
        self.num_cam = num_cam
        self.lora_config = lora_config
        self.lora_rank = lora_rank
        self.rgb_input = rgb_input
        self.rgb_img_size = rgb_img_size
        self.add_vision_id = add_vision_id
        self.tiled_rgb_imgs = tiled_rgb_imgs
        self.num_bins_actions = num_bins_actions
        self.use_flash_attention_2 = use_flash_attention_2
        self.action_mask_aug_per = action_mask_aug_per
        self.attention_dropout = attention_dropout

        self.model = self.load_qwen_model(
            qwen_model_id=self.qwen_model_id,
            use_lora=self.use_lora,
            use_qlora=self.use_qlora,
            lora_config=self.lora_config,
            lora_rank=self.lora_rank,
            use_flash_attention_2=self.use_flash_attention_2,
            attention_dropout=self.attention_dropout,
        )

        # Enable gradient checkpointing if requested
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

        self.min_pixel = self.max_pixel = self.rgb_img_size[0] * self.rgb_img_size[1]
        if self.rgb_input and self.tiled_rgb_imgs:
            self.min_pixel *= self.history * self.num_cam
            self.max_pixel *= self.history * self.num_cam

        self.processor = self.load_qwen_model_processor(
            qwen_model_id=qwen_model_id,
            min_pixel=self.min_pixel,
            max_pixel=self.max_pixel,
            padding_side="left" if use_flash_attention_2 else None,
        )
        self.logits_processor = NumberSpaceOnlyProcessor(self.processor.tokenizer)

        print(
            "WARNING: Using hardcoded dataset stats for DP3. This should be replaced with loading from a file."
        )

        if action_type == C.ORIGINAL:
            self.act_dim = original_action_dim
        elif action_type == C.EE:
            self.act_dim = 7
        else:
            assert False

        # ── CHANGED ──────────────────────────────────────────────────────────
        # Ask the model to write a Python function instead of a flat number
        # sequence.  The function `get_action(step_id)` accepts an integer in
        # [0, horizon-1] and returns a list of `act_dim` integers in
        # [0, num_bins_actions].  The model is encouraged to express the
        # trajectory analytically (e.g. sinusoidal, polynomial, exponential
        # easing) using numpy, math, or torch rather than a plain lookup table.
        self.system_message = (
            f"Analyze the input image and write a Python function named "
            f"'get_action' that takes a single integer argument 'step_id' "
            f"(ranging from 0 to {self.horizon - 1}) and returns a list of "
            f"{self.act_dim} integers (each clamped to [0, {self.num_bins_actions}]), "
            f"representing the predicted robot actions for that future timestep. "
            f"Express the trajectory analytically — use numpy, math, or torch to "
            f"compute smooth, physically plausible motion (e.g. sinusoidal curves, "
            f"polynomial easing, exponential approach, or spline-like interpolation). "
            f"Avoid plain static lookup tables. "
            f"Import any modules you need at the top of the function body. "
            f"Output only the Python function definition. Nothing else."
        )
        # ── END CHANGED ───────────────────────────────────────────────────────

        # Enough tokens for a full Python function with analytical trajectory code.
        # 1024 was too tight — the model would loop on comments and never emit a return.
        self.num_tokens = 4096
        self.original_dataset_stats = None  # original dataset stats has the same format as the one provided by the dataset
        self.dataset_stats = (
            None  # dataset stats is in the format specific to the model
        )

        self._sysuser_len = None

        self.cache_sysuser_len = False

        self._forward_count = 0

    def set_dataset_stats(self, dataset_stats):
        """
        Set the dataset stats for the model
        :param dataset_stats: dict, the dataset stats
        """
        if dataset_stats == {}:
            warnings.warn(
                "Dataset stats is empty likely because the system does not have the data used to compute the stats. Ignore this is you are loading a pretrained model."
            )
            return

        self.original_dataset_stats = dataset_stats
        if self.action_type == C.ORIGINAL:
            self.dataset_stats = dataset_stats["out_ori_act"]
        else:
            raise NotImplementedError(f"Action type {self.action_type} not implemented")

    @staticmethod
    def load_qwen_model(
        qwen_model_id,
        use_lora,
        use_qlora,
        lora_config,
        lora_rank,
        use_flash_attention_2,
        attention_dropout,
    ):
        if lora_config == "":
            lora_config = None
        elif lora_config == "default":
            if use_lora:
                from peft import LoraConfig, get_peft_model

                lora_config = LoraConfig(
                    lora_alpha=16,
                    lora_dropout=0.05,
                    r=lora_rank,
                    bias="none",
                    target_modules=["q_proj", "v_proj"],
                    task_type="CAUSAL_LM",
                )
        else:
            raise ValueError(f"Invalid lora_config: {lora_config}")

        # Qwen Init
        bnb_config = None
        if use_lora and use_qlora:
            from transformers import BitsAndBytesConfig

            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_type=torch.bfloat16,
            )

        extra_kwargs = {}
        if use_flash_attention_2:
            extra_kwargs["attn_implementation"] = "flash_attention_2"

        if attention_dropout > 0.0:
            config = AutoConfig.from_pretrained(qwen_model_id)
            config.attention_dropout = attention_dropout
            extra_kwargs["config"] = config

        model = Qwen3VLForConditionalGeneration.from_pretrained(
            qwen_model_id,
            # device_map={"": "cuda:0"},  # Use the explicit map
            quantization_config=bnb_config,
            torch_dtype=torch.bfloat16,
            # local_files_only=True,
            **extra_kwargs,
        )

        if use_lora and (lora_config is not None):
            model = get_peft_model(model, lora_config)

        return model

    @staticmethod
    def load_qwen_model_processor(
        qwen_model_id,
        min_pixel,
        max_pixel,
        padding_side,
    ):
        if padding_side is not None:
            processor = AutoProcessor.from_pretrained(
                qwen_model_id,
                min_pixels=min_pixel,
                max_pixels=max_pixel,
                padding_side=padding_side,
                # local_files_only=True,
            )
        else:
            processor = AutoProcessor.from_pretrained(
                qwen_model_id,
                min_pixels=min_pixel,
                max_pixels=max_pixel,
                # local_files_only=True,
            )

        return processor

    def get_min_max_act(self, instruction):
        """
        Get the min and max action for the instruction.
        :param instruction: str, the instruction for the current episode. This is needed for libero bounds 99% as the action space is different for different instructions.
        :return: torch.Tensor, the min and max action.
        """
        assert instruction is not None, "instruction is needed for libero bounds 99%"
        min_act = []
        max_act = []
        for _instruction in instruction:
            _suite = self.instruction_to_suite[_instruction]
            min_act.append(torch.tensor(self.dataset_stats[_suite]["min"]))
            max_act.append(torch.tensor(self.dataset_stats[_suite]["max"]))
        min_act = torch.stack(min_act, dim=0)
        max_act = torch.stack(max_act, dim=0)
        return min_act, max_act

    def get_text_action(self, actions, instruction=None):
        """
        Get the text action from the actions.
        :param actions: torch.Tensor, the actions to convert to text, shape (bs, horizon, act_dim).
        :param instruction: str, the instruction for the current episode.
        :return: List[str], one Python function string per sample in the batch.
        """
        # ── CHANGED ──────────────────────────────────────────────────────────
        # Previously this produced a flat space-separated string of integers.
        # Now it produces a Python function `get_action(step_id)` whose body
        # contains the full action table as a literal list-of-lists so the
        # ground-truth target matches the new generation format.
        if (
            (not hasattr(self, "_min_act"))
            or (not hasattr(self, "_max_act"))
            or (self._min_act.device != actions.device)
            or (self._max_act.device != actions.device)
        ):
            min_act = torch.tensor(self.dataset_stats["min"], device=actions.device)
            max_act = torch.tensor(self.dataset_stats["max"], device=actions.device)
            self._min_act = min_act
            self._max_act = max_act
        else:
            min_act = self._min_act
            max_act = self._max_act

        assert torch.all(min_act <= actions) and torch.all(
            actions <= max_act
        ), f"Action is out of range: {actions}"

        # Normalise → bin indices, shape (bs, horizon, act_dim)
        actions = (actions - min_act) / (max_act - min_act)
        actions *= self.num_bins_actions
        actions = torch.round(actions).long()  # (bs, horizon, act_dim)

        action_txt = []
        for sample_actions in actions:
            # sample_actions: (horizon, act_dim) integer tensor
            table = sample_actions.tolist()  # list[list[int]]
            fn = (
                "def get_action(step_id):\n"
                f"    actions = {table}\n"
                "    return actions[step_id]"
            )
            action_txt.append(fn)

        return action_txt
        # ── END CHANGED ───────────────────────────────────────────────────────

    def get_action_from_text_action(self, action_txt, instruction=None):
        """
        Get the action from the text action.
        :param action_txt: List[str], one Python function string per sample.
        :param instruction: str, the instruction for the current episode.
        :return: torch.Tensor of shape (bs, horizon, act_dim).
        """
        # ── CHANGED ──────────────────────────────────────────────────────────
        # Previously this parsed a flat space-separated integer string.
        # Now it exec-utes the generated `get_action(step_id)` function and
        # calls it for every step_id in [0, horizon-1], then de-normalises.
        bs = len(action_txt)
        min_act = torch.tensor(self.dataset_stats["min"])
        max_act = torch.tensor(self.dataset_stats["max"])

        results = []
        for txt in action_txt:
            try:
                fn_start = txt.find("def get_action")

                if fn_start != -1:
                    # ── new format: execute the generated Python function ──
                    # Strip markdown code fences the model may emit (```python ... ```)
                    txt_clean = txt
                    for fence in ("```python", "```"):
                        txt_clean = txt_clean.replace(fence, "")
                    fn_start = txt_clean.find("def get_action")
                    fn_src = txt_clean[fn_start:]
                    # Drop any top-level statements after the function body
                    # (e.g. `action = get_action(3)` / `print(...)` the model sometimes appends)
                    import ast as _ast
                    try:
                        tree = _ast.parse(fn_src)
                        # Keep only the function def node
                        fn_nodes = [n for n in tree.body if isinstance(n, _ast.FunctionDef)]
                        if fn_nodes:
                            fn_src = _ast.get_source_segment(fn_src, fn_nodes[0]) or fn_src
                    except Exception:
                        pass  # fall through to exec; it will raise a clearer error if needed
                    local_ns: dict = {}
                    exec(fn_src, {"__builtins__": __builtins__, "np": __import__("numpy"), "math": __import__("math"), "torch": __import__("torch")}, local_ns)  # noqa: S102
                    get_action_fn = local_ns["get_action"]

                    import numpy as _np

                    def _coerce(val, act_dim):
                        """Coerce any return value into a plain list of act_dim ints."""
                        if val is None:
                            return [0] * act_dim
                        # unwrap 0-d numpy arrays / torch tensors to a Python scalar
                        if isinstance(val, _np.ndarray):
                            val = val.flatten().tolist()
                        elif hasattr(val, "tolist"):  # torch.Tensor or similar
                            val = val.flatten().tolist()
                        # scalar → repeat across all dims
                        if not hasattr(val, "__iter__"):
                            val = [val] * act_dim
                        val = list(val)
                        val = val[:act_dim]
                        while len(val) < act_dim:
                            val.append(val[-1] if val else 0)
                        return [int(float(v)) for v in val]

                    step_actions = []
                    for step_id in range(self.horizon):
                        step = get_action_fn(step_id)
                        step_actions.append(_coerce(step, self.act_dim))

                    action = torch.tensor(step_actions, dtype=torch.float32)  # (horizon, act_dim)

                else:
                    # ── legacy fallback: flat space-separated integer string ──
                    tokens = txt.strip().split()
                    flat = [int(x) for x in tokens]
                    # make length divisible by act_dim
                    remainder = len(flat) % self.act_dim
                    if remainder != 0:
                        flat = flat[: len(flat) - remainder]
                    action = torch.tensor(flat, dtype=torch.float32).reshape(-1, self.act_dim)
                    # pad / trim to exactly horizon steps
                    if action.shape[0] < self.horizon:
                        action = torch.cat(
                            [action, action[-1:].repeat(self.horizon - action.shape[0], 1)],
                            dim=0,
                        )
                    action = action[: self.horizon]

                # de-normalise (shared by both paths)
                action = ((action / self.num_bins_actions) * (max_act - min_act)) + min_act

            except Exception as e:
                print(f"Error in parsing action function: {e}")
                print(txt)
                action = ((min_act + max_act) / 2).unsqueeze(0).repeat(self.horizon, 1)

            results.append(action)

        return torch.stack(results, dim=0)  # (bs, horizon, act_dim)
        # ── END CHANGED ───────────────────────────────────────────────────────

    def check_inputs(
        self,
        pc,
        rgb_pc,
        instr,
        rgb,
        ori_act,
        ee_pos,
        ee_rot,
        ee_gri,
        out_ee_pos,
        out_ee_rot,
        out_ee_gri,
        out_ori_act,
        get_loss,
        get_action,
        get_one_step_action,
        last_action_txt,
    ):
        assert (
            self.dataset_stats is not None
        ), "dataset_stats must be set before calling forward"
        assert isinstance(instr, list)

        assert not (get_loss and get_action)
        assert get_loss or get_action
        if get_one_step_action:
            assert get_action
            assert isinstance(last_action_txt, str)
            assert len(instr) == 1, "one_step_action is only supported for batch size 1"
        if self.rgb_input:
            assert rgb is not None
            assert rgb.shape[1:] == (
                self.history,
                self.num_cam,
                *self.rgb_img_size,
                3,
            ), f"rgb.shape: {rgb.shape}, self.history: {self.history}, self.num_cam: {self.num_cam}, self.rgb_img_size: {self.rgb_img_size}"
            # some room for numerical errors
            # 0, 2, 255 are valid values
            assert (rgb.min() >= -1e-2) and (
                1.99 <= rgb.max() <= 255.01
            ), f"rgb.min(): {rgb.min()}, rgb.max(): {rgb.max()}"
        else:
            assert rgb is None

        assert pc is None
        assert rgb_pc is None

    def get_imgs(
        self,
        bs,
        pc,
        rgb_pc,
        rgb,
    ):
        """
        Get the images for the given inputs
        :param bs: int, the batch size
        :param pc: torch.Tensor, the point cloud
        :param rgb_pc: torch.Tensor, the rgb point cloud
        :param rgb: torch.Tensor, the rgb image
        :return: list of list of PIL images
        """
        imgs = [[] for _ in range(bs)]

        if self.rgb_input:
            for i, _rgb in enumerate(rgb):
                _imgs = []
                for j in range(self.history):
                    for k in range(self.num_cam):
                        _imgs.append(_rgb[j][k])
                if self.tiled_rgb_imgs:
                    _imgs = [self.tile_images(_imgs)]
                _imgs = [
                    Image.fromarray(x.cpu().numpy().astype(np.uint8)) for x in _imgs
                ]
                imgs[i].extend(_imgs)

        return imgs

    def get_qwen_inputs(
        self,
        bs: int,
        imgs: List[List[PIL.Image.Image]],
        instr: List[str],
        action_txt: List[str],
        drop_assistant: bool = False,
        add_generation_prompt: bool = False,
    ):
        """
        Get the Qwen inputs for the given inputs
        :param bs: int, the batch size
        :param imgs: list of list of PIL images
        :param instr: list of strings
        :param action_txt: list of strings
        :param drop_assistant: bool, whether to drop the assistant portion.
        :param add_generation_prompt: bool, whether to add the generation prompt assistant\n.
        """

        examples = [
            format_data(
                system_message=self.system_message,
                image=imgs[i],
                instr=instr[i],
                action_txt=action_txt[i],
            )
            for i in range(bs)
        ]
        if drop_assistant:
            # drop the assistant portion so the model must generate it
            examples = [e[:2] for e in examples]

        texts = [
            self.processor.apply_chat_template(
                example,
                tokenize=False,
                add_generation_prompt=add_generation_prompt,
                add_vision_id=self.add_vision_id,
            )
            # when add_generation_prompt is True, it will add the prompt
            # `assistant\n` to the end of the input text
            for example in examples
        ]
        # [0] in process_vision_info is for image input, [1] is for video input
        image_inputs = [process_vision_info(example)[0] for example in examples]

        model_inputs = self.processor(
            text=texts, images=image_inputs, return_tensors="pt", padding=True
        )
        for key in model_inputs:
            model_inputs[key] = model_inputs[key].to(
                next(self.model.parameters()).device
            )
        return model_inputs, examples

    def forward(
        self,
        pc=None,
        rgb_pc=None,
        instr=None,
        rgb=None,
        ori_act=None,
        ee_pos=None,
        ee_rot=None,
        ee_gri=None,
        out_ee_pos=None,
        out_ee_rot=None,
        out_ee_gri=None,
        out_ori_act=None,
        get_loss=True,
        get_action=False,
        generate_temperature=0.1,
        get_one_step_action=False,
        last_action_txt="",
    ):
        """
        Forward pass for the Qwen model

        Parameters
        ----------
        pc : optional
            Point cloud data
        rgb_pc : optional
            RGB point cloud data
        instr : optional
            Instructions
        rgb : optional
            RGB images
        ori_act : optional
            Original actions
        ee_pos : optional
            End effector position
        ee_rot : optional
            End effector rotation
        ee_gri : optional
            End effector gripper state
        out_ee_pos : optional
            Output end effector position
        out_ee_rot : optional
            Output end effector rotation
        out_ee_gri : optional
            Output end effector gripper state
        out_ori_act : optional
            Output original actions
        get_loss : bool, default=True
            Whether to calculate and return loss
        get_action : bool, default=False
            Whether to get actions
        generate_temperature : float, default=0.1
            Temperature for generation
        get_one_step_action : bool, default=False
            Whether to run the model forward for only one step of action at a time. If True,
            we complete the last_action_txt to next action string sufficient to decode the
            action for one next step. If False, we complete the last_action_txt to next
            action string sufficient to decode the action for all the next steps.
        last_action_txt : str, default=""
            The last action text to complete to get the next action text. This is only
            used when get_one_step_action is True.

        Returns
        -------
        dict or tuple
            Model outputs (should specify exact return structure)

        Raises
        ------
        Exception
            Any exceptions that can be raised (if applicable)
        """
        self.check_inputs(
            pc=pc,
            rgb_pc=rgb_pc,
            instr=instr,
            rgb=rgb,
            ori_act=ori_act,
            ee_pos=ee_pos,
            ee_rot=ee_rot,
            ee_gri=ee_gri,
            out_ee_pos=out_ee_pos,
            out_ee_rot=out_ee_rot,
            out_ee_gri=out_ee_gri,
            out_ori_act=out_ori_act,
            get_loss=get_loss,
            get_action=get_action,
            get_one_step_action=get_one_step_action,
            last_action_txt=last_action_txt,
        )

        bs = len(instr)

        # imgs is list of list of PIL images
        imgs = self.get_imgs(
            bs=bs,
            pc=pc,
            rgb_pc=rgb_pc,
            rgb=rgb,
        )

        # TODO: implement this for ee action space
        if out_ori_act is None:
            assert not get_loss
            action_txt = [[]] * bs
        else:
            action_txt = self.get_text_action(out_ori_act, instruction=instr)

        model_inputs, examples = self.get_qwen_inputs(
            bs=bs,
            imgs=imgs,
            instr=instr,
            action_txt=action_txt,
            drop_assistant=get_action,  # when getting action, we drop the assistant portion
            add_generation_prompt=get_action,  # when getting action, we add the generation prompt assistant\n so that the model need not generate it
        )

        if get_loss:
            labels = model_inputs["input_ids"].clone()
            # mask system message and image token IDs in the labels
            for i, example in enumerate(examples):
                if (self._sysuser_len is None) or (not self.cache_sysuser_len):
                    sysuser_conv = example[:-1]
                    sysuser_text = self.processor.apply_chat_template(
                        sysuser_conv, tokenize=False, add_vision_id=self.add_vision_id
                    )
                    sysuser_img, _ = process_vision_info(sysuser_conv)

                    sysuser_inputs = self.processor(
                        text=[sysuser_text],
                        images=[sysuser_img],
                        return_tensors="pt",
                        padding=True,
                    )

                    sysuser_len = sysuser_inputs["input_ids"].shape[1]
                    sysuser_len += 3  # to mask out `assistant\n`
                    self._sysuser_len = sysuser_len
                else:
                    sysuser_len = self._sysuser_len
                # TIP: to decode the input use:
                # when padding is right: self.processor.decode(model_inputs["input_ids"][0][0:sysuser_len])
                # when padding is left: self.processor.decode(model_inputs["input_ids"][0][num_pad_tokens: num_pad_tokens + sysuser_len])
                if self.processor.tokenizer.padding_side == "right":
                    labels[i, :sysuser_len] = -100
                elif self.processor.tokenizer.padding_side == "left":
                    num_pad_tokens = sum(labels[i] == 151643).item()
                    labels[i, num_pad_tokens : num_pad_tokens + sysuser_len] = -100
                else:
                    raise ValueError(
                        f"Unknown padding side: {self.processor.tokenizer.padding_side}"
                    )

                assert (
                    not self.processor.tokenizer.padding_side == "left"
                ), "current implementation only supports right padding"
                # for debugging, compare
                # self.processor.decode(model_inputs["input_ids"][i][model_inputs["attention_mask"][i] == 1])
                # with self.processor.decode(model_inputs["input_ids"][i])
                _action_txt = action_txt[i]
                # 10% of sample has no augmentation
                if random.random() < 0.1:
                    _action_mask_aug_per = 0.0
                else:
                    _action_mask_aug_per = random.uniform(0.0, self.action_mask_aug_per)
                mask_len = int(len(_action_txt) * _action_mask_aug_per)
                mask_indices = random.sample(range(len(_action_txt)), mask_len)
                mask_indices = [
                    x + sysuser_len for x in mask_indices
                ]  # add sysuser_len to the mask indices to get the correct indices of these tokens
                labels[
                    i, mask_indices
                ] = -100  # these elements will not be used for loss calculation
                model_inputs["input_ids"][
                    i, mask_indices
                ] = 30  # replace the input ids with '?' token id

            labels[labels == 151643] = -100

            outputs = self.model(**model_inputs)
            logits = outputs.logits  # (batch_size, seq_len, vocab_size)

            # copied from modeling_qwen2_5_vl.py to compute the loss
            # Upcast to float if we need to compute the loss to avoid potential precision issues
            logits = logits.float()
            # Shift so that tokens < n predict n
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            # Flatten the tokens
            shift_logits = shift_logits.view(-1, shift_logits.size(-1))
            shift_labels = shift_labels.view(-1)
            # Enable model parallelism
            shift_labels = shift_labels.to(shift_logits.device)
            loss = self.loss_fn(shift_logits, shift_labels)

            return {"loss": loss}

        if get_action:
            sample_args = {}
            if generate_temperature > 0:
                sample_args["temperature"] = generate_temperature
            else:
                sample_args[
                    "do_sample"
                ] = False  # greedy search, this makes the generation deterministic

            if get_one_step_action:
                # we calculate the max number of tokens to generate for one step of action
                # +1 is for the space token
                max_new_tokens = self.act_dim * (len(str(self.num_bins_actions)) + 1)
                if last_action_txt != "":
                    last_action_txt_ids = self.processor.tokenizer(
                        last_action_txt, return_tensors="pt"
                    )["input_ids"].to(model_inputs["input_ids"].device)
                    model_inputs["input_ids"] = torch.cat(
                        [model_inputs["input_ids"], last_action_txt_ids], dim=1
                    )
                    model_inputs["attention_mask"] = torch.cat(
                        [
                            model_inputs["attention_mask"],
                            torch.ones_like(last_action_txt_ids),
                        ],
                        dim=1,
                    )

            # token id to text mapping
            # 220 is space
            # 15 to 24 are 0 to 9
            # 151645 is the end of text token
            # ── CHANGED ──────────────────────────────────────────────────────
            # logits_processor is removed: the model must now emit arbitrary
            # Python source code, so constraining to digits/spaces would
            # prevent it from generating keywords, brackets, etc.
            generated_ids = self.model.generate(
                **model_inputs,
                max_new_tokens=(
                    max_new_tokens if get_one_step_action else self.num_tokens
                ),
                **sample_args,
            )
            # ── END CHANGED ───────────────────────────────────────────────────

            input_ids = model_inputs["input_ids"]
            generated_ids_trimmed = [
                out_ids[len(in_ids) :]
                for in_ids, out_ids in zip(input_ids, generated_ids)
            ]
            generated_action_txt = self.processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )

            if get_one_step_action:
                # TODO: Only supports batch size 1
                generated_action_txt = [last_action_txt + generated_action_txt[0]]

            # ── CHANGED ──────────────────────────────────────────────────────
            self._forward_count += 1
            if self._forward_count % 50 == 0:
                print(
                    f"\n{'='*60}\n"
                    f"[Coding_QwenActor] Generated function @ forward pass {self._forward_count} "
                    f"(batch sample 0):\n"
                    f"{'─'*60}\n"
                    f"{generated_action_txt[0]}\n"
                    f"{'='*60}\n"
                )
            # ── END CHANGED ───────────────────────────────────────────────────

            out_ori_act = self.get_action_from_text_action(
                generated_action_txt, instruction=instr
            )

            if get_one_step_action:
                out_ori_act = out_ori_act[:, -1:]

            return {
                "gt_action_text": action_txt,
                "pred_action_txt": generated_action_txt,
                "gt_out_ori_act": out_ori_act,
                "out_ori_act": out_ori_act,
            }

    def save_pretrained(self, path):
        self.model.save_pretrained(path)
        self.processor.save_pretrained(path)

    def from_pretrained(self, path, is_trainable=True):
        print(f"[Coding_QwenActor] Skipping finetuned checkpoint '{path}' — using fresh pretrained weights from '{self.qwen_model_id}'.")
        return

        _device = next(self.parameters()).device
        # This way works
        del self.model
        torch.cuda.empty_cache()
        gc.collect()

        if self.use_lora:
            from peft import PeftModel

            base_model = self.load_qwen_model(
                qwen_model_id=self.qwen_model_id,
                use_lora=self.use_lora,
                use_qlora=self.use_qlora,
                lora_config="",  # None regarless of lora config as lora is added later using PeftModel
                lora_rank=self.lora_rank,  # Doesn't matter here as lora_config is None
                use_flash_attention_2=self.use_flash_attention_2,
                attention_dropout=self.attention_dropout,
            )

            self.model = PeftModel.from_pretrained(
                base_model,
                path,
                is_trainable=is_trainable,
            )
            print("Loading Qwen2.5 PEFT model from", path)
        else:
            extra_kwargs = {}
            if self.use_flash_attention_2:
                extra_kwargs["attn_implementation"] = "flash_attention_2"
            if self.attention_dropout > 0.0:
                config = AutoConfig.from_pretrained(path)
                config.attention_dropout = self.attention_dropout
                extra_kwargs["config"] = config
            self.model = Qwen3VLForConditionalGeneration.from_pretrained(
                path,
                # device_map={"": "cuda:0"},
                torch_dtype=torch.bfloat16,
                **extra_kwargs,
            )
            print("Loading Qwen2.5 full model from", path)

        if self.use_flash_attention_2:
            self.processor = AutoProcessor.from_pretrained(
                path,
                min_pixels=self.min_pixel,
                max_pixels=self.max_pixel,
                padding_side="left",
            )
        else:
            self.processor = AutoProcessor.from_pretrained(
                path,
                min_pixels=self.min_pixel,
                max_pixels=self.max_pixel,
            )

        print("Loading Qwen2.5 processor from", path)

        Coding_QwenActor.to(self, _device)

    def to(self, device):
        super().to(device)
        # if device is interger like 0 or "0", convert to cuda:0
        if isinstance(device, int) or (isinstance(device, str) and device.isnumeric()):
            device = f"cuda:{device}"
        if hasattr(self, "renderer"):
            self.renderer.renderer.device = device
            self.renderer.cameras.to(device)

    def tile_images(self, images):
        """
        Tile a images into a single image
        :param images: Tensor of shape (bs, H, W, 3) or list of tensors of shape (H, W, 3)
        :return: Tensor of shape (bs, H, W, 3)
        """
        for img in images:
            assert len(img.shape) == 3, f"img.shape: {img.shape}"
            assert img.shape[2] == 3, f"img.shape: {img.shape}"

        widths, heights = zip(*(im.shape[:-1] for im in images))
        total_width = sum(widths)
        max_height = max(heights)
        dst = torch.zeros((max_height, total_width, 3), device=images[0].device)
        current_x = 0
        for i, img in enumerate(images):
            dst[: img.shape[0], current_x : current_x + img.shape[1], :] = img
            current_x += img.shape[1]
        return dst

#### API based

In [ ]:
"""
API_Actor — VLA actor backed by any OpenAI-compatible chat/completions endpoint.

Both OpenRouter (https://openrouter.ai) and the HuggingFace Inference API
expose the standard /v1/chat/completions interface, so a single backend covers
every model: GPT-4o, Gemini 2.5 Pro, Llama, Gemma, Mistral, Qwen, …

Quick-start
───────────
    # via OpenRouter  (one key, every model)
    actor = API_Actor(
        base_url = "https://openrouter.ai/api/v1",
        model_id = "google/gemini-2.5-pro-preview",   # or "openai/gpt-4o", etc.
        api_key  = "sk-or-...",
        horizon=16, original_action_dim=7,
    )

    # via HuggingFace Inference API
    actor = API_Actor(
        base_url = "https://api-inference.huggingface.co/v1",
        model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct",
        api_key  = "hf_...",
        horizon=16, original_action_dim=7,
    )

    actor.set_dataset_stats(dataset_stats)
    out = actor.forward(instr=["pick up the cup"], rgb=rgb_tensor,
                        get_loss=False, get_action=True)
    actions = out["out_ori_act"]   # (bs, horizon, act_dim)

Convenience factory
───────────────────
    from api_actor import make_actor
    actor = make_actor("gemini-2.5-pro", horizon=16, original_action_dim=7)
    actor = make_actor("gpt-4o",         horizon=16, original_action_dim=7)
    actor = make_actor("llama-3.2-11b",  horizon=16, original_action_dim=7)
"""

import ast
import base64
import io
import json
import os
import warnings
from typing import List, Optional

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn

import rv_train.constants as C


# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _pil_to_b64(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


def _get_tokeniser():
    """tiktoken if available, else a simple char-level fallback."""
    try:
        import tiktoken
        return tiktoken.get_encoding("cl100k_base")
    except Exception:
        class _Char:
            def encode(self, t): return [ord(c) for c in t]
            n_vocab = 65536
        return _Char()


# ─────────────────────────────────────────────────────────────────────────────
# Preset catalogue  { short_name : (base_url, model_id) }
# ─────────────────────────────────────────────────────────────────────────────
# All entries use OpenRouter by default — swap base_url to the HF endpoint
# if you prefer to go directly to HuggingFace.

OR  = "https://openrouter.ai/api/v1"          # OpenRouter
HF  = "https://api-inference.huggingface.co/v1"  # HuggingFace

PRESETS: dict[str, tuple[str, str]] = {
    # ── OpenAI ────────────────────────────────────────────────────────────────
    "gpt-4o":             (OR, "openai/gpt-4o"),
    "gpt-4.1":            (OR, "openai/gpt-4.1"),
    "gpt-4.1-mini":       (OR, "openai/gpt-4.1-mini"),
    "o4-mini":            (OR, "openai/o4-mini"),
    # ── Google ────────────────────────────────────────────────────────────────
    "gemini-2.5-pro":     (OR, "google/gemini-2.5-pro-preview"),
    "gemini-2.0-flash":   (OR, "google/gemini-2.0-flash-001"),
    "gemma-3-27b":        (OR, "google/gemma-3-27b-it"),
    # ── Meta ──────────────────────────────────────────────────────────────────
    "llama-3.2-11b":      (HF, "meta-llama/Llama-3.2-11B-Vision-Instruct"),
    "llama-3.2-90b":      (HF, "meta-llama/Llama-3.2-90B-Vision-Instruct"),
    "llama-4-scout":      (OR, "meta-llama/llama-4-scout"),
    "llama-4-maverick":   (OR, "meta-llama/llama-4-maverick"),
    # ── Mistral ───────────────────────────────────────────────────────────────
    "pixtral-12b":        (OR, "mistralai/pixtral-12b"),
    "mistral-small-3.1":  (OR, "mistralai/mistral-small-3.1-24b-instruct"),
    # ── Qwen ──────────────────────────────────────────────────────────────────
    "qwen2.5-vl-72b":     (OR, "qwen/qwen2.5-vl-72b-instruct"),
    "qwen2.5-vl-7b":      (OR, "qwen/qwen2.5-vl-7b-instruct"),
    # ── DeepSeek ──────────────────────────────────────────────────────────────
    "deepseek-v3":        (OR, "deepseek/deepseek-chat-v3-0324"),
}


# ─────────────────────────────────────────────────────────────────────────────
# Main class
# ─────────────────────────────────────────────────────────────────────────────

class API_Actor(nn.Module):
    """
    Drop-in replacement for VLA0_QwenActor / Fresh_QwenActor / Coding_QwenActor.

    Uses a single OpenAI-compatible HTTP endpoint so the same code works with
    OpenRouter, the HuggingFace Inference API, a local vLLM server, etc.

    Parameters
    ----------
    base_url : str
        Root of the OpenAI-compatible API, e.g.
        "https://openrouter.ai/api/v1" or
        "https://api-inference.huggingface.co/v1".
    model_id : str
        Model string forwarded in the request body, e.g.
        "google/gemini-2.5-pro-preview" or
        "meta-llama/Llama-3.2-11B-Vision-Instruct".
    api_key : str, optional
        Falls back to OPENROUTER_API_KEY → HF_API_KEY → OPENAI_API_KEY.
    approx_loss : bool
        When True, forward(get_loss=True) computes an approximate
        token-level cross-entropy between the ground-truth action function
        string and the model's response (real logits are unavailable over
        a remote API).  Set False to raise NotImplementedError on training
        calls, which is safer if you only use this actor for evaluation.
    """

    def __init__(
        self,
        # ── new ──────────────────────────────────────────────────────────────
        base_url: str = OR,
        model_id: str = "google/gemini-2.5-pro-preview",
        api_key: Optional[str] = None,
        temperature: float = 0.1,
        max_retries: int = 3,
        approx_loss: bool = False,
        # ── same as local actors ─────────────────────────────────────────────
        qwen_model_id: str = "",          # ignored, kept for config compat
        action_type: str = C.ORIGINAL,
        original_action_dim: int = 7,
        horizon: int = 16,
        history: int = 1,
        use_lora: bool = False,
        use_qlora: bool = False,
        num_cam: int = 1,
        lora_config: str = "",
        lora_rank: int = 8,
        rgb_input: bool = True,
        rgb_img_size: tuple = (84, 84),
        add_vision_id: bool = False,
        tiled_rgb_imgs: bool = False,
        num_bins_actions: int = 1000,
        use_flash_attention_2: bool = False,
        system_message_version: int = 1,
        action_mask_aug: int = 0,
        action_mask_aug_per: float = 0.1,
        attention_dropout: float = 0.0,
    ):
        super().__init__()

        self.base_url        = base_url.rstrip("/")
        self.model_id        = model_id
        self.temperature     = temperature
        self.max_retries     = max_retries
        self.approx_loss     = approx_loss
        self.action_type     = action_type
        self.original_action_dim = original_action_dim
        self.horizon         = horizon
        self.history         = history
        self.num_cam         = num_cam
        self.rgb_input       = rgb_input
        self.rgb_img_size    = rgb_img_size
        self.tiled_rgb_imgs  = tiled_rgb_imgs
        self.num_bins_actions = num_bins_actions

        self.act_dim = (
            original_action_dim if action_type == C.ORIGINAL else
            7                   if action_type == C.EE        else
            (_ for _ in ()).throw(ValueError(f"Unknown action_type: {action_type}"))
        )

        self.api_key = (
            api_key
            or os.environ.get("OPENROUTER_API_KEY")
            or os.environ.get("HF_API_KEY")
            or os.environ.get("OPENAI_API_KEY")
            or ""
        )

        self.original_dataset_stats: Optional[dict] = None
        self.dataset_stats: Optional[dict] = None
        self._min_act: Optional[torch.Tensor] = None
        self._max_act: Optional[torch.Tensor] = None

        self.system_message = (
            f"Analyze the input image(s) and write a Python function named "
            f"'get_action' that takes a single integer argument 'step_id' "
            f"(ranging from 0 to {self.horizon - 1}) and returns a list of "
            f"{self.act_dim} integers (each clamped to [0, {self.num_bins_actions}]), "
            f"representing the predicted robot actions for that future timestep. "
            f"Express the trajectory analytically — use numpy or math to compute "
            f"smooth, physically plausible motion (sinusoidal curves, polynomial "
            f"easing, exponential approach, or spline-like interpolation). "
            f"Avoid plain static lookup tables. "
            f"Output only the Python function definition. Nothing else."
        )

        # tiny parameter so next(self.parameters()).device works
        self._device_anchor = nn.Parameter(torch.zeros(1), requires_grad=False)
        self._tokeniser = _get_tokeniser()
        self._forward_count = 0

        print(f"[API_Actor] {base_url}  model={model_id}")

    # ─────────────────────────────────────────────────────────────────────────
    # Dataset stats
    # ─────────────────────────────────────────────────────────────────────────

    def set_dataset_stats(self, dataset_stats: dict):
        if not dataset_stats:
            warnings.warn("Dataset stats is empty — skipping.")
            return
        self.original_dataset_stats = dataset_stats
        if self.action_type == C.ORIGINAL:
            self.dataset_stats = dataset_stats["out_ori_act"]
        else:
            raise NotImplementedError(f"action_type {self.action_type} not implemented")

    # ─────────────────────────────────────────────────────────────────────────
    # Checkpoint stubs
    # ─────────────────────────────────────────────────────────────────────────

    def save_pretrained(self, path: str):
        os.makedirs(path, exist_ok=True)
        cfg = dict(base_url=self.base_url, model_id=self.model_id,
                   action_type=self.action_type,
                   original_action_dim=self.original_action_dim,
                   horizon=self.horizon, history=self.history,
                   num_bins_actions=self.num_bins_actions)
        with open(os.path.join(path, "api_actor_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"[API_Actor] config saved → {path}")

    def from_pretrained(self, path: str, is_trainable: bool = True):
        p = os.path.join(path, "api_actor_config.json")
        if os.path.exists(p):
            with open(p) as f:
                cfg = json.load(f)
            self.base_url  = cfg.get("base_url",  self.base_url)
            self.model_id  = cfg.get("model_id",  self.model_id)
            print(f"[API_Actor] config loaded ← {p}")
        else:
            print(f"[API_Actor] no config at {p}, keeping current settings.")

    # ─────────────────────────────────────────────────────────────────────────
    # Forward
    # ─────────────────────────────────────────────────────────────────────────

    def forward(
        self,
        pc=None, rgb_pc=None, instr=None, rgb=None,
        ori_act=None, ee_pos=None, ee_rot=None, ee_gri=None,
        out_ee_pos=None, out_ee_rot=None, out_ee_gri=None, out_ori_act=None,
        get_loss: bool = True,
        get_action: bool = False,
        generate_temperature: Optional[float] = None,
        get_one_step_action: bool = False,
        last_action_txt: str = "",
    ):
        assert self.dataset_stats is not None, "Call set_dataset_stats() first."
        assert isinstance(instr, list)
        assert not (get_loss and get_action)
        assert get_loss or get_action

        temp = generate_temperature if generate_temperature is not None else self.temperature
        bs   = len(instr)
        imgs = self._get_pil_imgs(bs, rgb)

        action_txt = (
            [""] * bs if out_ori_act is None
            else self._get_text_action(out_ori_act)
        )

        # ── LOSS ─────────────────────────────────────────────────────────────
        if get_loss:
            if not self.approx_loss:
                raise NotImplementedError(
                    "Remote APIs do not expose token logits.  "
                    "Set approx_loss=True for the approximate loss, or use a "
                    "local actor for training."
                )
            losses = []
            for i in range(bs):
                pred = self._call_api(imgs[i], instr[i], temp)
                losses.append(self._token_ce(action_txt[i], pred))
            return {"loss": torch.stack(losses).mean()}

        # ── ACTION ────────────────────────────────────────────────────────────
        generated = []
        for i in range(bs):
            prefix = last_action_txt if (get_one_step_action and i == 0) else ""
            generated.append(self._call_api(imgs[i], instr[i], temp, prefix=prefix))

        self._forward_count += 1
        if self._forward_count % 20 == 0:
            print(
                f"\n{'='*60}\n"
                f"[API_Actor] {self.model_id}  pass #{self._forward_count}\n"
                f"{'─'*60}\n{generated[0]}\n{'='*60}\n"
            )

        out = self._get_action_from_text(generated)
        if get_one_step_action:
            out = out[:, -1:]

        return {
            "gt_action_text":  action_txt,
            "pred_action_txt": generated,
            "gt_out_ori_act":  out,
            "out_ori_act":     out,
        }

    # ─────────────────────────────────────────────────────────────────────────
    # API call  (single OpenAI-compatible endpoint)
    # ─────────────────────────────────────────────────────────────────────────

    def _call_api(
        self,
        imgs: List[Image.Image],
        instruction: str,
        temperature: float,
        prefix: str = "",
    ) -> str:
        import requests

        user_text = instruction
        if prefix:
            user_text = f"{instruction}\n\nPartial action so far:\n{prefix}\nContinue."

        # build multimodal content list
        content: list = []
        for img in imgs:
            content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{_pil_to_b64(img)}"},
            })
        content.append({"type": "text", "text": user_text})

        payload = {
            "model": self.model_id,
            "messages": [
                {"role": "system", "content": self.system_message},
                {"role": "user",   "content": content},
            ],
            "max_tokens": 4096,
            "temperature": max(temperature, 1e-4),
        }
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type":  "application/json",
        }

        last_exc = None
        for attempt in range(self.max_retries):
            try:
                r = requests.post(
                    f"{self.base_url}/chat/completions",
                    headers=headers,
                    json=payload,
                    timeout=120,
                )
                r.raise_for_status()
                return r.json()["choices"][0]["message"]["content"] or ""
            except Exception as e:
                last_exc = e
                print(f"[API_Actor] attempt {attempt+1}/{self.max_retries}: {e}")

        raise RuntimeError(f"API call failed after {self.max_retries} attempts") from last_exc

    # ─────────────────────────────────────────────────────────────────────────
    # Action ↔ text
    # ─────────────────────────────────────────────────────────────────────────

    def _ensure_minmax(self, device=None):
        if self._min_act is None:
            d = device or torch.device("cpu")
            self._min_act = torch.tensor(self.dataset_stats["min"], device=d)
            self._max_act = torch.tensor(self.dataset_stats["max"], device=d)
        return self._min_act, self._max_act

    def _get_text_action(self, actions: torch.Tensor) -> List[str]:
        mn, mx = self._ensure_minmax(actions.device)
        assert torch.all(mn <= actions) and torch.all(actions <= mx)
        bins = torch.round((actions - mn) / (mx - mn) * self.num_bins_actions).long()
        out = []
        for sample in bins:                        # (horizon, act_dim)
            table = sample.tolist()
            out.append(
                "def get_action(step_id):\n"
                f"    actions = {table}\n"
                "    return actions[step_id]"
            )
        return out

    def _get_action_from_text(self, action_txt: List[str]) -> torch.Tensor:
        mn, mx = self._ensure_minmax()
        results = []
        for txt in action_txt:
            try:
                action = self._parse_fn(txt)
            except Exception as e:
                print(f"[API_Actor] parse error: {e}\n{txt[:300]}")
                action = ((mn + mx) / 2).unsqueeze(0).repeat(self.horizon, 1)
            action = ((action / self.num_bins_actions) * (mx - mn)) + mn
            results.append(action)
        return torch.stack(results)               # (bs, horizon, act_dim)

    def _parse_fn(self, txt: str) -> torch.Tensor:
        for fence in ("```python", "```"):
            txt = txt.replace(fence, "")
        idx = txt.find("def get_action")
        if idx == -1:
            return self._parse_flat(txt)
        fn_src = txt[idx:]
        try:
            tree = ast.parse(fn_src)
            nodes = [n for n in tree.body if isinstance(n, ast.FunctionDef)]
            if nodes:
                fn_src = ast.get_source_segment(fn_src, nodes[0]) or fn_src
        except Exception:
            pass
        ns: dict = {}
        exec(fn_src, {                              # noqa: S102
            "__builtins__": __builtins__,
            "np":    __import__("numpy"),
            "math":  __import__("math"),
            "torch": __import__("torch"),
        }, ns)
        fn = ns["get_action"]
        steps = [self._coerce(fn(s)) for s in range(self.horizon)]
        return torch.tensor(steps, dtype=torch.float32)

    def _coerce(self, val) -> List[int]:
        if val is None:
            return [0] * self.act_dim
        if isinstance(val, np.ndarray):
            val = val.flatten().tolist()
        elif hasattr(val, "tolist"):
            val = val.flatten().tolist()
        if not hasattr(val, "__iter__"):
            val = [val] * self.act_dim
        val = list(val)[: self.act_dim]
        while len(val) < self.act_dim:
            val.append(val[-1] if val else 0)
        return [int(float(v)) for v in val]

    def _parse_flat(self, txt: str) -> torch.Tensor:
        flat = [int(x) for x in txt.strip().split()]
        rem  = len(flat) % self.act_dim
        if rem:
            flat = flat[:-rem]
        t = torch.tensor(flat, dtype=torch.float32).reshape(-1, self.act_dim)
        if t.shape[0] < self.horizon:
            t = torch.cat([t, t[-1:].repeat(self.horizon - t.shape[0], 1)])
        return t[: self.horizon]

    # ─────────────────────────────────────────────────────────────────────────
    # Approximate loss
    # ─────────────────────────────────────────────────────────────────────────

    def _token_ce(self, gt: str, pred: str) -> torch.Tensor:
        gt_ids   = self._tokeniser.encode(gt)
        pred_ids = self._tokeniser.encode(pred)
        n        = min(len(gt_ids), len(pred_ids), 512)
        if n == 0:
            return torch.tensor(10.0)
        vocab   = getattr(self._tokeniser, "n_vocab", 65536)
        gt_t    = torch.tensor(gt_ids[:n],   dtype=torch.long)
        pred_t  = torch.tensor(pred_ids[:n], dtype=torch.long)
        logits  = torch.zeros(n, vocab)
        logits.scatter_(1, pred_t.unsqueeze(1), 10.0)
        return F.cross_entropy(logits, gt_t)

    # ─────────────────────────────────────────────────────────────────────────
    # Image helpers
    # ─────────────────────────────────────────────────────────────────────────

    def _get_pil_imgs(self, bs: int, rgb) -> List[List[Image.Image]]:
        imgs = [[] for _ in range(bs)]
        if not self.rgb_input or rgb is None:
            return imgs
        for i, _rgb in enumerate(rgb):
            frames = []
            for j in range(self.history):
                for k in range(self.num_cam):
                    frames.append(_rgb[j][k])
            if self.tiled_rgb_imgs:
                frames = [self._tile(frames)]
            imgs[i] = [Image.fromarray(f.cpu().numpy().astype(np.uint8)) for f in frames]
        return imgs

    def _tile(self, images):
        W = sum(i.shape[1] for i in images)
        H = max(i.shape[0] for i in images)
        dst, x = torch.zeros((H, W, 3), device=images[0].device), 0
        for img in images:
            dst[:img.shape[0], x:x + img.shape[1]] = img
            x += img.shape[1]
        return dst


# ─────────────────────────────────────────────────────────────────────────────
# Convenience factory
# ─────────────────────────────────────────────────────────────────────────────

def make_actor(preset: str, **kwargs) -> API_Actor:
    """
    Create an API_Actor from a short preset name.

    Examples
    --------
    >>> actor = make_actor("gemini-2.5-pro", horizon=16, original_action_dim=7)
    >>> actor = make_actor("llama-3.2-11b",  horizon=16, original_action_dim=7)
    >>> actor = make_actor("gpt-4o",         horizon=16, original_action_dim=7)
    """
    if preset not in PRESETS:
        raise ValueError(f"Unknown preset '{preset}'. Available: {sorted(PRESETS)}")
    base_url, model_id = PRESETS[preset]
    return API_Actor(base_url=base_url, model_id=model_id, **kwargs)
    
    
    
    
    
    
    """
actor_registry.py
─────────────────
Central place that maps a short name → a QwenActor-compatible class.

Usage in your notebook / train script
──────────────────────────────────────
    from actor_registry import QwenActor        # drop-in for Fresh_QwenActor etc.

    # ── or pick explicitly ────────────────────────────────────────────────────
    from actor_registry import (
        VLA0Actor,          # original flat-number model
        FreshActor,         # Qwen2.5 + Python-function output
        CodingActor,        # Qwen3-VL + Python-function output
        GPT4oActor,         # OpenAI  GPT-4o   via OpenRouter
        Gemini25Actor,      # Google  Gemini 2.5 Pro via OpenRouter
        Llama4Actor,        # Meta    Llama-4 Maverick via OpenRouter
        Qwen72bActor,       # Qwen    2.5-VL-72B via OpenRouter
        HFLlamaActor,       # Meta    Llama-3.2-11B direct from HuggingFace
    )

All API-backed actors are functools.partial objects that pre-fill
(base_url, model_id) so the remaining kwargs are identical to the
local actors — get_model() works without any changes.
"""

import functools
import os

# ─────────────────────────────────────────────────────────────────────────────
# Endpoint constants
# ─────────────────────────────────────────────────────────────────────────────
_OR = "https://openrouter.ai/api/v1"           # one key → every model
_HF = "https://api-inference.huggingface.co/v1"  # free tier for small models


def _api(base_url: str, model_id: str, **extra_defaults):
    """
    Return a partial of API_Actor with (base_url, model_id) pre-filled.
    The result has the exact same __init__ signature as the local actors,
    so it slots straight into get_model().
    """
    return functools.partial(
        API_Actor,
        base_url=base_url,
        model_id=model_id,
        **extra_defaults,
    )


# ─────────────────────────────────────────────────────────────────────────────
# API-backed actor aliases
# ─────────────────────────────────────────────────────────────────────────────

# ── OpenAI (via OpenRouter) ───────────────────────────────────────────────────
GPT4oActor       = _api(_OR, "openai/gpt-4o")
GPT41Actor       = _api(_OR, "openai/gpt-4.1")
GPT41MiniActor   = _api(_OR, "openai/gpt-4.1-mini")
O4MiniActor      = _api(_OR, "openai/o4-mini")

# ── Google (via OpenRouter) ───────────────────────────────────────────────────
Gemini25Actor    = _api(_OR, "google/gemini-2.5-pro-preview")
Gemini20Actor    = _api(_OR, "google/gemini-2.0-flash-001")
Gemma27bActor    = _api(_HF, "google/gemma-3-27b-it")

# ── Meta (via OpenRouter) ─────────────────────────────────────────────────────
Llama4Actor      = _api(_OR, "meta-llama/llama-4-maverick")   # best Meta vision+code
Llama4ScoutActor = _api(_OR, "meta-llama/llama-4-scout")      # cheaper

# ── Qwen (via OpenRouter) — best vision+code OSS choice ──────────────────────
Qwen72bActor     = _api(_OR, "qwen/qwen2.5-vl-72b-instruct")
Qwen7bActor      = _api(_OR, "qwen/qwen2.5-vl-7b-instruct")

# ── Mistral (via OpenRouter) ──────────────────────────────────────────────────
PixtralActor     = _api(_OR, "mistralai/pixtral-12b")

# ── DeepSeek (via OpenRouter, text-only — good for ablations without vision) ──
DeepSeekActor    = _api(_OR, "deepseek/deepseek-chat-v3-0324")

# ── Direct HuggingFace (models not on OpenRouter) ────────────────────────────
HFLlamaActor     = _api(_HF, "meta-llama/Llama-3.2-11B-Vision-Instruct")
HFLlama90bActor  = _api(_HF, "meta-llama/Llama-3.2-90B-Vision-Instruct")


# ─────────────────────────────────────────────────────────────────────────────
# *** THIS IS THE ONE LINE YOU CHANGE IN YOUR NOTEBOOK ***
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:

QwenActor = Coding_QwenActor # or Fresh_QwenActor or Coding_QwenActor or VLA0_QwenActor
# Gemma27bActor

# ─────────────────────────────────────────────────────────────────────────────
# Pretty-printer so you can see what is active
# ─────────────────────────────────────────────────────────────────────────────
def which_actor():
    if isinstance(QwenActor, functools.partial):
        print(f"[actor_registry] QwenActor → API_Actor "
              f"base_url={QwenActor.keywords['base_url']}  "
              f"model={QwenActor.keywords['model_id']}")
    else:
        print(f"[actor_registry] QwenActor → {QwenActor.__name__} (local)")



# factory functions

In [ ]:
import argparse
import gc
import os

import torch
from roboverse.evals.libero.eval import eval, get_evaluation_tasks

import argparse
import gc
import os
import pickle as pkl
import pprint
import random
import shutil
from contextlib import redirect_stdout
from datetime import datetime
from time import time

import roboverse

import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import tqdm
from torch import autocast
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader

from rv_train.configs import get_cfg_defaults


def get_optimizer(cfg, model, num_gpus=1):
    """
    Returns optimizer and learning rate scheduler based on the config
    :param cfg: config object
    :param model: model to optimize
    :param num_gpus: number of GPUs to optimize the model on, required for scaling the learning rate
    """
    if cfg.EXP.OPTIMIZER == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(), lr=cfg.TRAIN.lr * num_gpus, weight_decay=cfg.TRAIN.l2
        )
    elif cfg.EXP.OPTIMIZER == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=cfg.TRAIN.lr * num_gpus, weight_decay=cfg.TRAIN.l2
        )
    elif cfg.EXP.OPTIMIZER == "adam_bnb":
        import bitsandbytes as bnb

        optimizer = bnb.optim.Adam(
            model.parameters(), lr=cfg.TRAIN.lr * num_gpus, weight_decay=cfg.TRAIN.l2
        )
    elif cfg.EXP.OPTIMIZER == "adamw_bnb":
        import bitsandbytes as bnb

        optimizer = bnb.optim.AdamW(
            model.parameters(), lr=cfg.TRAIN.lr * num_gpus, weight_decay=cfg.TRAIN.l2
        )
    elif cfg.EXP.OPTIMIZER == "adamw_bnb_fp8":
        import bitsandbytes as bnb

        optimizer = bnb.optim.AdamW8bit(
            model.parameters(), lr=cfg.TRAIN.lr * num_gpus, weight_decay=cfg.TRAIN.l2
        )
    else:
        raise NotImplementedError

    if cfg.EXP.LR_SCHED == "none":
        lr_sched = None
    elif cfg.EXP.LR_SCHED == "cosine_anneal":
        lr_sched = lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg.TRAIN.num_epochs, eta_min=cfg.LR_SCHED.lr_clip
        )
    else:
        raise NotImplementedError

    return optimizer, lr_sched


def load_model(model, model_path, cfg):
    """
    Loads a pretrained model from a given path.
    :param model: model to load
    :param model_path: path to the pretrained model
    :param cfg: config object
    """
    print(f"Recovering model and checkpoint from {model_path}")
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)

    # take care of DDP
    if isinstance(model, DDP):
        model_module = model.module
    else:
        model_module = model

    # take care of model loading for models that have load_pretrained method
    if hasattr(model_module, "from_pretrained"):
        print("WARNING: model has from_pretrained method")
        assert model_path[-4:] == ".pth"
        print(f"Loading from {model_path[:-4]}")
        model_module.from_pretrained(model_path[:-4])
    else:
        model_module.load_state_dict(checkpoint["model_state"])

    # load the dataset stats
    if cfg.EXP.MODEL in ["qwen", "dp", "qwen_dp"]:
        log_dir = "/".join(model_path.split("/")[:-1])
        with open(f"{log_dir}/dataset_stats.pkl", "rb") as f:
            original_dataset_stats = pkl.load(f)
            model_module.set_dataset_stats(original_dataset_stats)

    return model, checkpoint

def load_model_opt_sched(
    model,
    optimizer,
    lr_sched,
    model_path,
    cfg,
    to_load_model=True,
    only_load_model=False,
):
    """
    Loads a pretrained model from a given path.
    :param model: model to load
    :param optimizer: optimizer to load
    :param lr_sched: learning rate scheduler to load
    :param model_path: path to the pretrained model
    :param cfg: config object
    :param to_load_model: whether to load the model from the checkpoint or not
    """
    if to_load_model:
        model, checkpoint = load_model(model, model_path, cfg)
    else:
        checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)

    if not only_load_model:
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        if lr_sched is not None:
            lr_sched.load_state_dict(checkpoint["lr_sched_state"])

    epoch = checkpoint["epoch"]

    # clean GPU memory
    torch.cuda.empty_cache()
    gc.collect()
    return model, epoch, optimizer, lr_sched



def get_cfg(cfg_path, cfg_opts):
    cfg = get_cfg_defaults()
    if cfg_path != "":
        cfg.merge_from_file(cfg_path)

    if cfg_opts != "":
        cfg.merge_from_list(cfg_opts.split(" "))
        cfg.EXP.EXP_ID += f"_{utils.short_name(cfg_opts)}"
    cfg.freeze()

    print(cfg)
    return cfg



def get_model(cfg, calculate_dataset_stats=True):
    """
    Returns model based on the config
    """
    if cfg.EXP.MODEL == "qwen":
        model = QwenActor(**cfg.MODEL.QWEN)
    else:
        assert False, f"Invalid model: {cfg.EXP.MODEL}"

    if calculate_dataset_stats and cfg.EXP.MODEL in ["qwen"]:
        temp_dataset = get_dataloader(split="train", cfg=cfg, get_dataset=True)
        model.set_dataset_stats(temp_dataset.stats)
        del temp_dataset

    return model



def get_pretrained_model(model_path, device, torch_compile=False):
    """
    Loads a pretrained model from a given path.
    :param model_path: path to the pretrained model
    :param device: device to load the model on, supports only single GPU for now
    :return: model, cfg
    """
    model_folder = "/".join(model_path.split("/")[:-1])
    cfg_path = model_folder + "/config.yaml"
    cfg = get_cfg(cfg_path, cfg_opts="")

    model = get_model(
        cfg, calculate_dataset_stats=False
    )  # don't calculate dataset stats for pretrained model, its loaded from a checkpoint
    model.to(device)
    optimizer, lr_sched = get_optimizer(cfg, model, num_gpus=1)

    model, _, _, _ = load_model_opt_sched(
        model=model,
        optimizer=optimizer,
        lr_sched=lr_sched,
        model_path=model_path,
        cfg=cfg,
        only_load_model=True,
    )

    if torch_compile:
        print(
            "Compiling model with torch.compile, this will put the model in eval mode and may take a while..."
        )
        model.eval()
        model = torch.compile(model)
        if hasattr(model, "model"):
            if hasattr(model.model, "generate"):
                print("Compiling model.model.generate with torch.compile")
                model.model.generate = torch.compile(model.model.generate)

    return model, cfg

# Run Eval

In [ ]:


# def main():



from types import SimpleNamespace

# ── Mirrors the argparse defaults in eval/eval_libero.py ─────
# args = SimpleNamespace(
#     # passed in cli command
#     model_path = "/kaggle/working/vla0/runs/vla0/model_last.pth",
#     task_suite_name = "libero_goal",
#     task_name = "put_the_wine_bottle_on_top_of_the_cabinet",
#     action_horizon = 1,
#     ensemble_prediction = 8,
#     task_id_count = 10,
#     task_id_index = 0,


#     # default configs from argparser
#     start_seed = 7,
#     save_all_data = True,
#     not_skip_evaluated = True,
#     amp = True,
#     generate_temperature = 0.,
#     ensemble_version = 1,
#     no_torch_compile = True,
#     num_steps = 0,
#     ensemble_2_weight = .5,
# )


from tqdm.notebook import tqdm
import roboverse.evals.libero.eval as eval_module
eval_module.tqdm = tqdm
# lightweight and fast for ~smoke test
args = SimpleNamespace(
    model_path = "/kaggle/working/vla0/runs/vla0/model_last.pth",
    task_suite_name = "libero_goal",
    task_name = "put_the_wine_bottle_on_top_of_the_cabinet",
    
    action_horizon = 8,       # was 1 — execute 8 actions before next inference (8x fewer model calls)
    ensemble_prediction = 1,  # was 8 — single forward pass instead of 8 (8x speedup per inference)
    task_id_count = 2,        # was 10 — only eval 2 episodes instead of 10
    task_id_index = 0,

    start_seed = 7,
    save_all_data = False,    # was True — skip saving data to disk
    not_skip_evaluated = True,
    amp = True,
    generate_temperature = 0.,
    ensemble_version = 1,
    no_torch_compile = True,
    num_steps = 100,          # was 0 (unlimited) — cap each episode at 100 steps
    ensemble_2_weight = .5,
)



all_tasks = get_evaluation_tasks()
assert (
    args.task_suite_name in all_tasks
), f"Task suite {args.task_suite_name} not found in {all_tasks.keys()}"
assert (
    args.task_name in all_tasks[args.task_suite_name]
), f"Task {args.task_name} not found in {all_tasks[args.task_suite_name]}"

model, cfg = get_pretrained_model(
    args.model_path, 0, torch_compile=not args.no_torch_compile
)
model.eval()

assert (
    cfg.EXP.DATASET == "roboverse"
), f"Dataset is {cfg.EXP.DATASET}, not roboverse"

assert cfg.EXP.MODEL in [
    "qwen",
], f"Model is {cfg.EXP.MODEL}, not qwen, if expanding must take care of action_type and action_horizon"

action_type = {
    "qwen": cfg.MODEL.QWEN.action_type,
}[cfg.EXP.MODEL]

if args.action_horizon == 0:
    action_horizon = {
        "qwen": cfg.MODEL.QWEN.horizon,
    }[cfg.EXP.MODEL]
else:
    action_horizon = args.action_horizon

enable_amp = args.amp
other_args = {}
if cfg.EXP.MODEL == "qwen":
    other_args["generate_temperature"] = args.generate_temperature

def model_act(*args, **kwargs):
    with torch.no_grad():
        with torch.autocast(
            device_type="cuda", dtype=torch.bfloat16, enabled=enable_amp
        ):
            out = model(  # noqa
                *args,
                **kwargs,
                **other_args,
                get_loss=False,
                get_action=True,
            )
            return out

log_dir = f"{args.model_path}"
if args.action_horizon != 0:
    log_dir = f"{log_dir}_ah_{args.action_horizon}"
if args.amp:
    log_dir = f"{log_dir}_amp"
if args.generate_temperature > 0:
    log_dir = f"{log_dir}_gen_temp_{args.generate_temperature}"
if args.ensemble_prediction > 1:
    log_dir = f"{log_dir}_ens_pred_{args.ensemble_prediction}"
if args.ensemble_version > 1:
    log_dir = f"{log_dir}_ens_ver_{args.ensemble_version}"
    if args.ensemble_version == 2 and args.ensemble_2_weight != 0.5:
        log_dir = f"{log_dir}_ens_2_weight_{args.ensemble_2_weight}"
if args.num_steps > 0:
    log_dir = f"{log_dir}_num_steps_{args.num_steps}"
log_dir = f"{log_dir}_eval_libero"

log_dir = os.path.join(log_dir, args.task_suite_name)
log_dir = os.path.join(log_dir, args.task_name)
os.makedirs(log_dir, exist_ok=True)

eval(
    model=model_act,
    action_type=action_type,
    cfg_path=cfg.DATALOADER.ROBOVERSE.cfg_path,
    cfg_opts=cfg.DATALOADER.ROBOVERSE.cfg_opts,
    task_name=args.task_name,
    task_suite_name=args.task_suite_name,
    log_dir=log_dir,
    save_video=True,
    seed=args.start_seed,
    action_horizon=action_horizon,
    skip_evaluated=not args.not_skip_evaluated,
    save_all_data=args.save_all_data,
    ensemble_prediction=args.ensemble_prediction,
    ensemble_2_weight=args.ensemble_2_weight,
    ensemble_version=args.ensemble_version,
    task_id_index=args.task_id_index,
    task_id_count=args.task_id_count,
    num_steps=args.num_steps,
)

del model
del model_act
gc.collect()


# if __name__ == "__main__":
# main()